# Unsloth Fine-tuning Lab on Kaggle: the minimalist LLM factory behind your Cloudflare Tunnel

Fine-tune small open LLMs on a free Kaggle GPU with [Unsloth](https://github.com/unslothai/unsloth) (4-bit QLoRA, about 2x faster and much less memory than plain Hugging Face training), from a web page or from an agent:

1. **Dataset:** upload JSONL / JSON / CSV (chat messages, ShareGPT, Alpaca, prompt/completion or raw text), import from the Hugging Face Hub, or let an agent add examples.
2. **Train:** pick a base model (Llama 3.2, Qwen2.5, Gemma 2, Mistral, or any Hub id), watch loss and eval loss live, stop early any time (the adapter is still saved).
3. **Test:** chat with the base model and your fine-tuned runs side by side; see the model's answers on held-out examples.
4. **Ship:** download the LoRA adapter, export a merged 16-bit model or GGUF (`q4_k_m`, `q8_0`, `f16` for Ollama, llama.cpp, LM Studio), or push to the Hub.

| Path | What |
|---|---|
| `/` | The lab page |
| `/api/*` | JSON API (dataset, training, runs, exports, chat) |
| `/mcp` | MCP server for agents (tools: `list_base_models`, `dataset_info`, `add_examples`, `import_hf_dataset`, `start_training`, `training_status`, `stop_training`, `list_runs`, `get_run`, `chat`, `export_run`, `export_status`, `push_to_hub`, `delete_run`) |

### Before you run
1. **Accelerator:** GPU T4 x2: training on GPU 0, chat and exports on GPU 1. (One GPU works; chat and exports then wait for training.) **Internet:** on.
2. **A Cloudflare Tunnel** with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then tick them for this notebook):
   - `UNSLOTH_TUNNEL_TOKEN`: the tunnel token. Without it everything still runs, only without a public URL.
   - `UNSLOTH_UI_PASSWORD` (recommended): browser login is any username + this password; agents send `Authorization: Bearer <password>`. Without it a random password is printed below.
   - `HF_TOKEN` (optional): for gated base models (Llama) and for pushing to the Hub (needs write access).
4. **Run All.** The install takes about 3–5 minutes. A 3B model on 1,000 short examples trains in roughly 10–20 minutes on a T4.

Runs (datasets, adapters, metrics) are kept in `/kaggle/working/unsloth_lab`; exports are large and go to `/tmp/unsloth_lab/exports`. Both disappear when the session ends: download or push what you want to keep. Logs: `/kaggle/working/logs/`.

In [ ]:
# 1. Settings
PORT = 7860
WORK_DIR = "/kaggle/working/unsloth_lab"
EXPORT_DIR = "/tmp/unsloth_lab/exports"    # merged models are 3-16 GB: outside /kaggle/working's 20 GB
LOG_DIR = "/kaggle/working/logs"
APP_DIR = '/kaggle/working/unsloth_app'
print("Settings saved.")

### 2. App files
These cells only write files into `APP_DIR`.

In [ ]:
# Shared web kit: password, MCP endpoint, proxy, process helpers
# Writes studio_http.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'studio_http.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""studio_http: the small web kit behind the newer agentic-notebooks apps.

build_notebook.py copies this file into each notebook's app folder, so every notebook stays
self-contained: nothing is imported from GitHub at run time. Standard library only.

An app built on it gets:
- routes with regex paths, JSON in and out, file (with byte ranges), zip and chunked responses;
- one password for everything: browsers get a login page and a signed session cookie (no browser pop-up);
  agents send ``Authorization: Bearer <password>``, ``X-Access-Token: <password>`` or HTTP Basic.
  ``GET /health`` stays open;
- an MCP server at ``POST /mcp`` (Streamable HTTP transport, plain JSON responses, no sessions),
  and the same tools as REST: ``GET /mcp/tools`` and ``POST /mcp/tools/<name>`` with the arguments as JSON;
- an optional reverse proxy (HTTP and WebSocket) for every path no route matches;
- IPv4 and IPv6 listeners on one port (cloudflared may dial [::1] for "localhost").

The notebook cells use the helpers at the bottom: kaggle_secret(), spawn() / stop_process() with pid
files (so re-running a cell, or the whole notebook after a kernel restart, never leaves a second copy
running), wait_http(), tail(), start_tunnel() and keep_alive(). Logs go to files, never to the notebook.
"""
import base64
import hashlib
import hmac
import html as _html
import http.client
import json
import logging
import mimetypes
import os
import re
import select
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, quote, urlparse

MCP_VERSIONS = ("2025-06-18", "2025-03-26", "2024-11-05")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailer", "trailers",
       "transfer-encoding", "upgrade"}
SECRET_HEADERS = {"authorization", "x-access-token"}
SESSION_COOKIE = "studio_session"
SESSION_DAYS = 30
PID_DIR = "/kaggle/working/.pids" if os.path.isdir("/kaggle/working") else "/tmp/agentic-notebooks-pids"


# ====================================================================== responses
class HTTPError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


class Response:
    def __init__(self, body=b"", status=200, ctype="text/plain; charset=utf-8", headers=None):
        self.body = body.encode("utf-8") if isinstance(body, str) else body
        self.status, self.ctype, self.headers = status, ctype, dict(headers or {})


class FileResponse:
    """A file from disk. ``name`` makes it a download. Single byte ranges work (audio/video seeking)."""

    def __init__(self, path, name=None, ctype=None, cache="no-cache"):
        if not os.path.isfile(path):
            raise HTTPError(404, "file not found")
        self.path, self.name, self.cache = path, name, cache
        self.ctype = ctype or mimetypes.guess_type(path)[0] or "application/octet-stream"


class StreamResponse:
    """Chunked response from an iterable of bytes or str."""

    def __init__(self, chunks, ctype="text/plain; charset=utf-8", headers=None):
        self.chunks, self.ctype, self.headers = chunks, ctype, dict(headers or {})


class ZipResponse:
    """Streams ``folder`` (or a list of (path, arcname) pairs) as a zip download."""

    def __init__(self, files, name):
        if isinstance(files, str):
            root = os.path.dirname(os.path.abspath(files))
            pairs = []
            for dp, dn, fn in os.walk(files):
                dn.sort()
                for f in sorted(fn):
                    if not f.endswith((".tmp", ".part")):
                        pairs.append((os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), root)))
            files = pairs
        self.files, self.name = files, name


class ToolContent(list):
    """Return this from an MCP tool to send raw content blocks, for example an image."""


def text_block(text):
    return {"type": "text", "text": text}


def image_block(data, mime="image/png"):
    return {"type": "image", "data": base64.b64encode(data).decode("ascii"), "mimeType": mime}


def file_logger(path, name="app"):
    """A logger that writes to ``path`` only (nothing reaches the notebook output)."""
    log = logging.getLogger(name)
    path = os.path.abspath(path)
    if not any(getattr(h, "baseFilename", None) == path for h in log.handlers):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        h = logging.FileHandler(path, encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S"))
        log.addHandler(h)
    log.setLevel(logging.INFO)
    log.propagate = False
    return log


def _disposition(name):
    ascii_name = re.sub(r'[^A-Za-z0-9_.() -]+', "_", name) or "download"
    return "attachment; filename=\"%s\"; filename*=UTF-8''%s" % (ascii_name, quote(name))


# ====================================================================== requests
class Request:
    def __init__(self, handler, params, max_body):
        self.handler = handler
        u = urlparse(handler.path)
        self.method = handler.command
        self.path = u.path
        self.query = {k: v[-1] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        self.params = params
        self.headers = handler.headers
        self.max_body = max_body
        self.consumed = False

    @property
    def length(self):
        try:
            return int(self.headers.get("Content-Length") or 0)
        except ValueError:
            raise HTTPError(400, "bad Content-Length")

    @property
    def chunked(self):
        return "chunked" in (self.headers.get("Transfer-Encoding") or "").lower()

    def has_body(self):
        return self.chunked or self.length > 0

    def iter_body(self, limit=None):
        """Yields the request body in pieces (Content-Length or chunked), enforcing ``limit`` bytes."""
        limit = self.max_body if limit is None else limit
        too_big = HTTPError(413, "request body too large (limit %d MB)" % (limit // 2 ** 20))
        rf = self.handler.rfile
        self.consumed = True
        if self.chunked:
            total = 0
            while True:
                line = rf.readline(1024)
                try:
                    size = int(line.split(b";")[0].strip() or b"0", 16)
                except ValueError:
                    raise HTTPError(400, "bad chunked body")
                if size == 0:
                    while rf.readline(1024) not in (b"\r\n", b"\n", b""):
                        pass
                    return
                total += size
                if total > limit:
                    raise too_big
                left = size
                while left:
                    data = rf.read(min(left, 1 << 20))
                    if not data:
                        raise HTTPError(400, "request body was cut off")
                    left -= len(data)
                    yield data
                rf.readline(8)
        else:
            left = self.length
            if left > limit:
                raise too_big
            while left:
                data = rf.read(min(left, 1 << 20))
                if not data:
                    raise HTTPError(400, "request body was cut off")
                left -= len(data)
                yield data

    def body(self):
        return b"".join(self.iter_body())

    def json(self):
        raw = self.body()
        if not raw.strip():
            return {}
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            raise HTTPError(400, "body is not valid JSON")

    def save_body(self, path, limit):
        """Writes the raw body to ``path`` (atomically) and returns its size."""
        if not self.has_body():
            raise HTTPError(411, "send the file as the raw request body")
        tmp = path + ".part"
        size = 0
        try:
            with open(tmp, "wb") as f:
                for data in self.iter_body(limit):
                    f.write(data)
                    size += len(data)
        except BaseException:
            if os.path.exists(tmp):
                os.remove(tmp)
            raise
        os.replace(tmp, path)
        return size

    def arg(self, name, default=None, cast=str):
        v = self.query.get(name)
        if v in (None, ""):
            return default
        try:
            return cast(v)
        except (TypeError, ValueError):
            raise HTTPError(400, "bad value for %s" % name)


# ====================================================================== app
def _strip_session(cookie):
    """Never forward our session cookie to a proxied backend."""
    return "; ".join(p.strip() for p in (cookie or "").split(";")
                     if p.strip() and p.strip().partition("=")[0] != SESSION_COOKIE)


LOGIN_PAGE = """<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Sign in · {{NAME}}</title>
<style>
:root{--bg:#f4f5f8;--card:#fff;--text:#14171c;--muted:#636b77;--line:#dfe2e8;--accent:#2f6fec;--bad:#c4372f}
@media (prefers-color-scheme:dark){:root{--bg:#0d0f13;--card:#161a20;--text:#e8eaee;--muted:#98a0ab;--line:#2a2f37;--accent:#5b8ff5;--bad:#ef6a60}}
*{box-sizing:border-box}html,body{height:100%;margin:0}
body{background:radial-gradient(1200px 600px at 10% -10%,color-mix(in srgb,var(--accent) 18%,transparent),transparent),var(--bg);
color:var(--text);font:15px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,"Noto Sans","Noto Sans Devanagari",sans-serif;
display:flex;align-items:center;justify-content:center;padding:16px}
.card{width:100%;max-width:380px;background:var(--card);border:1px solid var(--line);border-radius:18px;padding:30px 28px;
box-shadow:0 20px 60px rgba(0,0,0,.12)}
.logo{width:44px;height:44px;border-radius:12px;background:var(--accent);display:flex;align-items:center;justify-content:center;
color:#fff;font-weight:800;font-size:20px;margin-bottom:18px}
h1{font-size:20px;margin:0 0 4px}p{color:var(--muted);margin:0 0 22px;font-size:14px}
label{display:block;font-size:13px;color:var(--muted);margin-bottom:6px}
.field{position:relative}
input{width:100%;font:inherit;color:var(--text);background:transparent;border:1px solid var(--line);border-radius:10px;padding:11px 44px 11px 12px}
input:focus{outline:2px solid color-mix(in srgb,var(--accent) 45%,transparent);border-color:var(--accent)}
.eye{position:absolute;right:6px;top:50%;transform:translateY(-50%);border:0;background:none;color:var(--muted);cursor:pointer;padding:6px;font-size:13px}
button.go{width:100%;margin-top:16px;border:0;border-radius:10px;padding:12px;background:var(--accent);color:#fff;font:600 15px system-ui;cursor:pointer}
button.go:disabled{opacity:.6}.err{color:var(--bad);font-size:13px;min-height:20px;margin-top:10px}
.hint{font-size:12px;color:var(--muted);margin-top:18px;border-top:1px solid var(--line);padding-top:14px}
</style></head><body>
<form class="card" id="f" method="post" action="/login">
<div class="logo">&#9656;</div><h1>{{NAME}}</h1><p>Sign in with the password from your notebook.</p>
<label for="pw">Password</label><div class="field"><input id="pw" name="password" type="password" autocomplete="current-password" autofocus required>
<button class="eye" type="button" id="eye">Show</button></div>
<button class="go" id="go">Sign in</button><div class="err" id="err"></div>
<div class="hint">It is the <b>*_UI_PASSWORD</b> Kaggle secret, or the password the notebook printed.</div>
</form>
<script>
const f=document.getElementById('f'),pw=document.getElementById('pw'),err=document.getElementById('err'),go=document.getElementById('go');
document.getElementById('eye').onclick=e=>{pw.type=pw.type==='password'?'text':'password';e.target.textContent=pw.type==='password'?'Show':'Hide';};
f.onsubmit=async e=>{e.preventDefault();go.disabled=true;err.textContent='';
 try{const r=await fetch('/login',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({password:pw.value})});
  if(r.ok){location.reload();return;} const d=await r.json().catch(()=>({}));err.textContent=d.error||('Sign-in failed ('+r.status+')');}
 catch(x){err.textContent='Network error: '+x;} go.disabled=false;pw.select();};
</script></body></html>"""


def _rpc_error(mid, code, message):
    return {"jsonrpc": "2.0", "id": mid, "error": {"code": code, "message": message}}


class App:
    def __init__(self, name, password=None, version="1.0", instructions="", proxy=None, log=None,
                 max_body=64 << 20):
        self.name, self.version, self.instructions = name, version, instructions
        self.password = password or None
        self.proxy = proxy.rstrip("/") if proxy else None
        self.log = log or logging.getLogger(name)
        self.max_body = max_body
        self.routes, self.tools, self.servers = [], {}, []
        self._fails = {}

        self.route("GET", "/health", auth=False)(lambda req: {"ok": True})
        self.route("POST", "/login", auth=False)(self._login)
        self.route("GET", "/logout", auth=False)(self._logout)
        self.route("POST", "/logout", auth=False)(self._logout)
        self.route("POST", "/mcp")(self._mcp_http)
        self.route("GET", "/mcp")(lambda req: Response(
            "This MCP server answers POST requests (Streamable HTTP transport, JSON responses).", 405,
            headers={"Allow": "POST"}))
        self.route("DELETE", "/mcp")(lambda req: Response(b"", 405, headers={"Allow": "POST"}))
        self.route("GET", "/mcp/tools")(lambda req: {"tools": self.tool_list()})
        self.route("POST", r"/mcp/tools/(?P<name>[A-Za-z0-9_\-]+)")(self._tool_http)

    # ------------------------------------------------------------------ registration
    def route(self, method, pattern, auth=True):
        rx = re.compile(pattern)

        def deco(fn):
            self.routes.append((method.upper(), rx, fn, auth))
            return fn
        return deco

    def page(self, path, file):
        self.route("GET", re.escape(path))(lambda req: FileResponse(file, ctype="text/html; charset=utf-8"))

    def static(self, prefix, folder):
        root = os.path.realpath(folder)

        def serve(req):
            full = os.path.realpath(os.path.join(root, req.params["rel"]))
            if not full.startswith(root + os.sep):
                raise HTTPError(404, "not found")
            return FileResponse(full)
        self.route("GET", re.escape(prefix) + r"(?P<rel>.+)")(serve)

    def tool(self, name, description, properties=None, required=()):
        """Registers an MCP tool. ``properties`` is a JSON Schema properties object."""
        def deco(fn):
            self.tools[name] = {"fn": fn, "schema": {
                "name": name, "description": description,
                "inputSchema": {"type": "object", "properties": properties or {}, "required": list(required)}}}
            return fn
        return deco

    def tool_list(self):
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name, args):
        """Runs a tool. Returns (content_blocks, is_error)."""
        t = self.tools[name]
        if not isinstance(args, dict):
            return [text_block("arguments must be a JSON object")], True
        try:
            out = t["fn"](**args)
        except TypeError as e:
            if "argument" in str(e):
                return [text_block("bad arguments for %s: %s" % (name, e))], True
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("TypeError: %s" % e)], True
        except HTTPError as e:
            return [text_block(str(e))], True
        except (ValueError, KeyError) as e:
            return [text_block(str(e).strip("'\""))], True
        except Exception as e:
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("%s: %s" % (type(e).__name__, e))], True
        if isinstance(out, ToolContent):
            return list(out), False
        if isinstance(out, str):
            return [text_block(out)], False
        return [text_block(json.dumps(out, indent=1, ensure_ascii=False, default=str))], False

    # ------------------------------------------------------------------ MCP
    def _mcp_http(self, req):
        try:
            msg = json.loads(req.body().decode("utf-8") or "null")
        except ValueError:
            return _rpc_error(None, -32700, "parse error")
        if isinstance(msg, list):
            out = [r for r in map(self._rpc, msg) if r is not None]
            return out if out else Response(b"", 202)
        r = self._rpc(msg)
        return r if r is not None else Response(b"", 202)

    def _rpc(self, m):
        if not isinstance(m, dict) or m.get("jsonrpc") != "2.0":
            return _rpc_error(m.get("id") if isinstance(m, dict) else None, -32600, "invalid request")
        if "method" not in m or "id" not in m:
            return None            # a notification, or a response to a request we never send
        mid, method, params = m["id"], m["method"], m.get("params") or {}
        try:
            if method == "initialize":
                v = params.get("protocolVersion")
                result = {"protocolVersion": v if v in MCP_VERSIONS else MCP_VERSIONS[0],
                          "capabilities": {"tools": {"listChanged": False}},
                          "serverInfo": {"name": self.name, "version": self.version}}
                if self.instructions:
                    result["instructions"] = self.instructions
            elif method == "ping":
                result = {}
            elif method == "tools/list":
                result = {"tools": self.tool_list()}
            elif method == "tools/call":
                name = params.get("name")
                if name not in self.tools:
                    return _rpc_error(mid, -32602, "unknown tool: %s" % name)
                content, err = self.call_tool(name, params.get("arguments") or {})
                result = {"content": content, "isError": err}
            elif method == "resources/list":
                result = {"resources": []}
            elif method == "resources/templates/list":
                result = {"resourceTemplates": []}
            elif method == "prompts/list":
                result = {"prompts": []}
            else:
                return _rpc_error(mid, -32601, "method not found: %s" % method)
        except Exception as e:
            self.log.error("mcp %s failed\n%s", method, traceback.format_exc())
            return _rpc_error(mid, -32603, str(e))
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    def _tool_http(self, req):
        name = req.params["name"]
        if name not in self.tools:
            raise HTTPError(404, "unknown tool: %s" % name)
        content, err = self.call_tool(name, req.json())
        if len(content) == 1 and content[0]["type"] == "text":
            try:
                data = json.loads(content[0]["text"])
            except ValueError:
                data = content[0]["text"]
            body = {"error": data} if err else data
        else:
            body = {"content": content, "isError": err}
        return Response(json.dumps(body, ensure_ascii=False, default=str), 400 if err else 200,
                        "application/json; charset=utf-8")

    # ------------------------------------------------------------------ auth
    def _key(self):
        return hashlib.sha256(("studio-session:" + self.password).encode("utf-8")).digest()

    def make_session(self, days=SESSION_DAYS):
        exp = str(int(time.time() + days * 86400))
        return "%s.%s" % (exp, hmac.new(self._key(), exp.encode(), hashlib.sha256).hexdigest()[:40])

    def _session_ok(self, headers):
        for part in (headers.get("Cookie") or "").split(";"):
            k, _, v = part.strip().partition("=")
            if k == SESSION_COOKIE:
                exp, _, sig = v.partition(".")
                if exp.isdigit() and int(exp) > time.time():
                    want = hmac.new(self._key(), exp.encode(), hashlib.sha256).hexdigest()[:40]
                    if hmac.compare_digest(sig.encode(), want.encode()):
                        return True
        return False

    def _cookie(self, headers, value, max_age):
        secure = (headers.get("X-Forwarded-Proto") or "").lower() == "https" or '"https"' in (headers.get("Cf-Visitor") or "")
        return "%s=%s; Path=/; Max-Age=%d; HttpOnly; SameSite=Lax%s" % (
            SESSION_COOKIE, value, max_age, "; Secure" if secure else "")

    def _login(self, req):
        if not self.password:
            return {"ok": True}
        ip = req.headers.get("Cf-Connecting-Ip") or req.handler.client_address[0]
        now = time.time()
        fails = [t for t in self._fails.get(ip, []) if now - t < 300]
        if len(fails) >= 8:
            raise HTTPError(429, "too many attempts; wait a few minutes")
        if "json" in (req.headers.get("Content-Type") or ""):
            given = str(req.json().get("password") or "")
        else:
            given = parse_qs(req.body().decode("utf-8", "replace")).get("password", [""])[0]
        if not hmac.compare_digest(given.encode("utf-8"), self.password.encode("utf-8")):
            fails.append(now)
            self._fails[ip] = fails
            time.sleep(0.5)
            raise HTTPError(401, "wrong password")
        self._fails.pop(ip, None)
        return Response(json.dumps({"ok": True}), 200, "application/json; charset=utf-8",
                        {"Set-Cookie": self._cookie(req.headers, self.make_session(), SESSION_DAYS * 86400)})

    def _logout(self, req):
        return Response(b"", 303, "text/plain", {"Location": "/", "Set-Cookie": self._cookie(req.headers, "", 0)})

    def authorized(self, headers):
        pw = self.password
        if not pw:
            return True
        if self._session_ok(headers):
            return True

        def ok(given):
            return bool(given) and hmac.compare_digest(given.encode("utf-8"), pw.encode("utf-8"))
        if ok(headers.get("X-Access-Token", "")):
            return True
        h = headers.get("Authorization", "")
        if h[:6].lower() == "basic ":
            try:
                given = base64.b64decode(h[6:].strip()).decode("utf-8").partition(":")[2]
            except Exception:
                given = ""
            return ok(given)
        if h[:7].lower() == "bearer ":
            return ok(h[7:].strip())
        return False

    def _deny(self, h):
        """Pages get the login form; everything else a JSON 401 (no browser pop-up)."""
        if h.command in ("GET", "HEAD") and "text/html" in (h.headers.get("Accept") or ""):
            return self._respond(h, Response(LOGIN_PAGE.replace("{{NAME}}", _html.escape(self.name)), 401,
                                             "text/html; charset=utf-8"))
        self._respond(h, Response(json.dumps({"error": "login required: sign in on the page, or send "
                                              "'Authorization: Bearer <password>'"}), 401,
                                  "application/json; charset=utf-8"))

    # ------------------------------------------------------------------ dispatch
    def dispatch(self, h):
        h._sent = False
        req = None
        try:
            path = urlparse(h.path).path
            method = "GET" if h.command == "HEAD" else h.command
            found, path_known = None, False
            for m, rx, fn, auth in self.routes:
                mt = rx.fullmatch(path)
                if mt:
                    path_known = True
                    if m == method:
                        found = (fn, auth, mt.groupdict())
                        break
            if found is None:
                if path_known:
                    raise HTTPError(405, "method not allowed")
                if not self.proxy:
                    raise HTTPError(404, "not found")
                if not self.authorized(h.headers):
                    return self._deny(h)
                return self._proxy(h)
            fn, auth, params = found
            if auth and not self.authorized(h.headers):
                return self._deny(h)
            req = Request(h, params, self.max_body)
            self._respond(h, fn(req))
        except (BrokenPipeError, ConnectionResetError):
            h.close_connection = True
        except HTTPError as e:
            self._fail(h, e.status, str(e))
        except (ValueError, KeyError) as e:
            msg = ("missing field %s" % e) if isinstance(e, KeyError) else str(e)
            self._fail(h, 400, msg)
        except Exception as e:
            self.log.error("%s %s failed\n%s", h.command, h.path, traceback.format_exc())
            self._fail(h, 500, "%s: %s" % (type(e).__name__, e))
        finally:
            body_left = (req is None or not req.consumed) and (
                "chunked" in (h.headers.get("Transfer-Encoding") or "").lower()
                or (h.headers.get("Content-Length") or "0").strip() not in ("", "0"))
            if body_left:
                h.close_connection = True   # an unread body would corrupt the next request on this connection

    def _fail(self, h, status, message):
        if h._sent:
            h.close_connection = True
            return
        try:
            self._respond(h, Response(json.dumps({"error": message}), status, "application/json; charset=utf-8"))
        except OSError:
            h.close_connection = True

    def _start(self, h, status, headers):
        h.send_response(status)
        h._sent = True
        for k, v in headers:
            h.send_header(k, v)
        h.end_headers()

    def _respond(self, h, out):
        head = h.command == "HEAD"
        if out is None:
            out = Response(b"", 204)
        elif isinstance(out, (dict, list)):
            out = Response(json.dumps(out, ensure_ascii=False, default=str), 200, "application/json; charset=utf-8")
        elif isinstance(out, str):
            out = Response(out)
        elif isinstance(out, bytes):
            out = Response(out, 200, "application/octet-stream")

        if isinstance(out, Response):
            hdrs = {"Content-Type": out.ctype, "Cache-Control": "no-store"}
            hdrs.update(out.headers)
            if out.status not in (204, 304):
                hdrs["Content-Length"] = str(len(out.body))
            self._start(h, out.status, hdrs.items())
            if not head and out.status not in (204, 304) and out.body:
                h.wfile.write(out.body)
        elif isinstance(out, FileResponse):
            self._send_file(h, out, head)
        elif isinstance(out, StreamResponse):
            self._start(h, 200, list({"Content-Type": out.ctype, "Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no", **out.headers}.items())
                        + [("Transfer-Encoding", "chunked")])
            if head:
                h.wfile.write(b"0\r\n\r\n")
                return
            for chunk in out.chunks:
                if isinstance(chunk, str):
                    chunk = chunk.encode("utf-8")
                if chunk:
                    h.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                    h.wfile.flush()
            h.wfile.write(b"0\r\n\r\n")
        elif isinstance(out, ZipResponse):
            self._send_zip(h, out, head)
        else:
            raise TypeError("handler returned %r" % type(out))

    def _send_file(self, h, f, head):
        size = os.path.getsize(f.path)
        start, end, status = 0, size - 1, 200
        m = re.fullmatch(r"bytes=(\d*)-(\d*)", (h.headers.get("Range") or "").strip())
        if m and size and (m.group(1) or m.group(2)):
            a, b = m.groups()
            if a == "":
                start = max(0, size - int(b))
            else:
                start, end = int(a), min(int(b), size - 1) if b else size - 1
            if start > end or start >= size:
                return self._start(h, 416, [("Content-Range", "bytes */%d" % size), ("Content-Length", "0")])
            status = 206
        hdrs = [("Content-Type", f.ctype), ("Content-Length", str(end - start + 1)), ("Accept-Ranges", "bytes"),
                ("Cache-Control", f.cache)]
        if status == 206:
            hdrs.append(("Content-Range", "bytes %d-%d/%d" % (start, end, size)))
        if f.name:
            hdrs.append(("Content-Disposition", _disposition(f.name)))
        self._start(h, status, hdrs)
        if head:
            return
        with open(f.path, "rb") as fh:
            fh.seek(start)
            left = end - start + 1
            while left > 0:
                data = fh.read(min(left, 1 << 20))
                if not data:
                    break
                h.wfile.write(data)
                left -= len(data)

    def _send_zip(self, h, z, head):
        self._start(h, 200, [("Content-Type", "application/zip"), ("Content-Disposition", _disposition(z.name)),
                             ("Cache-Control", "no-store"), ("Transfer-Encoding", "chunked")])
        w = h.wfile

        class Chunks:
            def write(self, data):
                if data:
                    w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
                return len(data)

            def flush(self):
                w.flush()

        if not head:
            with zipfile.ZipFile(Chunks(), "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
                for path, arc in z.files:
                    if isinstance(path, bytes):
                        zf.writestr(arc, path)
                    elif os.path.isfile(path):
                        zf.write(path, arc, compress_type=zipfile.ZIP_STORED if path.endswith(
                            (".safetensors", ".gguf", ".bin", ".npy", ".png", ".jpg", ".mp4", ".zip", ".wav"))
                            else zipfile.ZIP_DEFLATED)
        w.write(b"0\r\n\r\n")
        w.flush()

    # ------------------------------------------------------------------ reverse proxy
    def _proxy(self, h):
        pu = urlparse(self.proxy)
        host, port = pu.hostname, pu.port or 80
        if ("upgrade" in (h.headers.get("Connection") or "").lower()
                and (h.headers.get("Upgrade") or "").lower() == "websocket"):
            return self._proxy_ws(h, host, port)
        req = Request(h, {}, self.max_body)
        headers = [(k, _strip_session(v) if k.lower() == "cookie" else v) for k, v in h.headers.items()
                   if k.lower() not in HOP and k.lower() not in SECRET_HEADERS and k.lower() != "content-length"]
        headers = [(k, v) for k, v in headers if v]
        headers.append(("X-Forwarded-For", h.client_address[0]))
        body = req.body() if req.has_body() else None
        if body is not None:
            headers.append(("Content-Length", str(len(body))))
        conn = http.client.HTTPConnection(host, port, timeout=900)
        try:
            # the original Host header is kept: some backends (ComfyUI) compare it with Origin
            conn.putrequest(h.command, h.path, skip_host="host" in {k.lower() for k, v in headers},
                            skip_accept_encoding=True)
            for k, v in headers:
                conn.putheader(k, v)
            conn.endheaders(body)
            r = conn.getresponse()
        except OSError as e:
            conn.close()
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        try:
            hdrs = [(k, v) for k, v in r.getheaders() if k.lower() not in HOP and k.lower() != "content-length"]
            cl = r.getheader("Content-Length")
            chunked = "chunked" in (r.getheader("Transfer-Encoding") or "").lower()
            if h.command == "HEAD" or r.status in (204, 304) or r.status < 200:
                return self._start(h, r.status, hdrs + ([("Content-Length", cl)] if cl else []))
            if cl is not None and not chunked:
                self._start(h, r.status, hdrs + [("Content-Length", cl)])
                while True:
                    data = r.read(1 << 16)
                    if not data:
                        break
                    h.wfile.write(data)
            else:
                self._start(h, r.status, hdrs + [("Transfer-Encoding", "chunked")])
                while True:
                    data = r.read1(1 << 16)
                    if not data:
                        break
                    h.wfile.write(b"%x\r\n" % len(data) + data + b"\r\n")
                    h.wfile.flush()
                h.wfile.write(b"0\r\n\r\n")
        finally:
            conn.close()

    def _proxy_ws(self, h, host, port):
        try:
            up = socket.create_connection((host, port), timeout=10)
        except OSError as e:
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        lines = ["%s %s HTTP/1.1" % (h.command, h.path)]
        lines += ["%s: %s" % (k, _strip_session(v) if k.lower() == "cookie" else v) for k, v in h.headers.items()
                  if k.lower() not in SECRET_HEADERS and (k.lower() != "cookie" or _strip_session(v))]
        h._sent = True
        h.close_connection = True
        client = h.connection
        try:
            up.sendall(("\r\n".join(lines) + "\r\n\r\n").encode("latin-1"))
            up.settimeout(None)
            client.settimeout(None)
            while True:
                ready, _, broken = select.select([client, up], [], [client, up], 600)
                if broken:
                    break
                for s in ready:
                    data = s.recv(1 << 16)
                    if not data:
                        return
                    (up if s is client else client).sendall(data)
        except OSError:
            pass
        finally:
            up.close()

    # ------------------------------------------------------------------ serving
    def handler_class(self):
        app = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = re.sub(r"\W+", "", app.name) or "studio"

            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                app.dispatch(self)

            do_POST = do_PUT = do_PATCH = do_DELETE = do_HEAD = do_OPTIONS = do_GET
        return Handler

    def start(self, port):
        """Listens on 0.0.0.0 and [::] in background threads. Calling it again restarts cleanly."""
        self.stop()
        handler = self.handler_class()
        for cls, host in ((_V4Server, "0.0.0.0"), (_V6Server, "::")):
            try:
                srv = cls((host, port), handler)
            except OSError as e:
                self.log.warning("could not listen on [%s]:%d: %s", host, port, e)
                continue
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            self.servers.append(srv)
        if not self.servers:
            raise RuntimeError("nothing is listening on port %d" % port)
        self.log.info("%s listening on port %d", self.name, port)

    def stop(self):
        while self.servers:
            srv = self.servers.pop()
            try:
                srv.shutdown()
                srv.server_close()
            except Exception:
                pass

    def serve_forever(self, port):
        """For app processes started with spawn(): serve until SIGTERM."""
        done = threading.Event()
        signal.signal(signal.SIGTERM, lambda *a: done.set())
        signal.signal(signal.SIGINT, lambda *a: done.set())
        self.start(port)
        done.wait()
        self.log.info("stopping")
        self.stop()


class _V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
    request_queue_size = 64


class _V6Server(_V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


# ====================================================================== JSON files and jobs
def read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path, data):
    """Atomic write, so readers in other threads or processes never see half a file."""
    tmp = "%s.%d.%d.tmp" % (path, os.getpid(), threading.get_ident())
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def safe_name(s, n=60):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")[:n]


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            i, n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": n, "mem_used_mb": int(u), "mem_total_mb": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


# ====================================================================== notebook helpers
def kaggle_secret(name):
    """Value of a Kaggle secret, or None when it is missing or not attached to this notebook.
    Outside Kaggle (local runs, Docker) it reads the environment variable of the same name."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get(name) or None
    try:
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def ui_password(secret_name):
    pw = kaggle_secret(secret_name)
    if pw:
        print("UI password: from the %s secret. Browser login: any username + that password." % secret_name)
        return pw
    import secrets
    pw = secrets.token_urlsafe(9)
    print("No %s secret, so this session's password is: %s" % (secret_name, pw))
    print("Browser login: any username + that password. Add the secret to keep a fixed password.")
    return pw


def _pidfile(name):
    return os.path.join(PID_DIR, safe_name(name) + ".json")


def _alive(pid, exe=None):
    try:
        with open("/proc/%d/stat" % pid) as f:
            if f.read().rsplit(")", 1)[1].split()[0] == "Z":
                return False
        if exe:
            with open("/proc/%d/cmdline" % pid, "rb") as f:
                return exe.encode() in f.read()
        return True
    except (OSError, IndexError):
        return False


def process_alive(name):
    info = read_json(_pidfile(name))
    return bool(info) and _alive(info["pid"], info.get("exe"))


def stop_process(name, timeout=20):
    """Stops a process spawn() started under ``name`` (and its children). Returns True if one was running."""
    info = read_json(_pidfile(name))
    if not info:
        return False
    pid, exe = info["pid"], info.get("exe")
    running = _alive(pid, exe)
    if running:
        for sig, wait in ((signal.SIGTERM, timeout), (signal.SIGKILL, 5)):
            try:
                os.killpg(pid, sig)
            except OSError:
                break
            t = time.time()
            while time.time() - t < wait and _alive(pid, exe):
                time.sleep(0.2)
            if not _alive(pid, exe):
                break
    try:
        os.remove(_pidfile(name))
    except OSError:
        pass
    return running


def spawn(name, cmd, log_path, env=None, cwd=None):
    """Starts ``cmd`` in the background with its output appended to ``log_path``.
    A process started earlier under the same name is stopped first, so re-running a cell is safe."""
    stop_process(name)
    os.makedirs(PID_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(os.path.abspath(log_path)), exist_ok=True)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s: starting %s =====\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), name)).encode())
        log.flush()
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                env=env, cwd=cwd, start_new_session=True)
    write_json(_pidfile(name), {"pid": proc.pid, "exe": os.path.basename(cmd[0]), "log": log_path,
                                "started": time.time()})
    return proc


def wait_http(url, timeout=120, name=None, headers=None):
    """Waits until ``url`` answers (any HTTP status). Returns False on timeout or if process ``name`` died."""
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=3)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            pass
        if name and not process_alive(name):
            return False
        time.sleep(1)
    return False


def tail(path, n=30):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 64 * 1024))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-n:])
    except OSError:
        return "(no log yet at %s)" % path


def ensure_cloudflared(path="/kaggle/working/cloudflared"):
    if not os.path.exists(path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
        os.chmod(path, 0o755)
    return path


def start_tunnel(secret_name, port=7860, log_path="/kaggle/working/logs/cloudflared.log",
                 binary="/kaggle/working/cloudflared"):
    """Starts cloudflared with the token from ``secret_name`` (passed as TUNNEL_TOKEN, never printed).
    A missing secret is not an error: the app keeps running inside the session, without a public URL."""
    stop_process("cloudflared")
    token = kaggle_secret(secret_name)
    if not token:
        print("No public URL: there is no Kaggle secret named %s attached to this notebook.\n"
              "The app is running inside this session anyway. To publish it:\n"
              "  1. Cloudflare Zero Trust -> Networks -> Tunnels: create a tunnel and add a public hostname\n"
              "     pointing to http://localhost:%d\n"
              "  2. Kaggle: Add-ons -> Secrets -> add %s with the tunnel token, and tick it for this notebook\n"
              "  3. Re-run this cell." % (secret_name, port, secret_name))
        return None
    ensure_cloudflared(binary)
    offset = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    proc = spawn("cloudflared", [binary, "tunnel", "--no-autoupdate", "run"], log_path,
                 env=dict(os.environ, TUNNEL_TOKEN=token))

    def new_log():
        with open(log_path, "rb") as f:
            f.seek(offset)
            return f.read().decode("utf-8", "replace")
    for _ in range(45):
        time.sleep(1)
        if "Registered tunnel connection" in new_log():
            print("Tunnel connected. Open the public hostname you routed to http://localhost:%d" % port)
            return proc
        if proc.poll() is not None:
            print("cloudflared stopped (exit code %s). Last log lines:" % proc.returncode)
            print("\n".join(new_log().splitlines()[-15:]))
            print("Check that %s holds the token of a tunnel that still exists." % secret_name)
            return None
    print("Tunnel not registered yet; it keeps trying in the background. Last log lines:")
    print("\n".join(new_log().splitlines()[-15:]))
    return proc


def keep_alive(port, names, extra=None, interval=60):
    """The last cell: prints one status line a minute so the Kaggle session stays active.
    Interrupting it only stops the printing; the app keeps running."""
    try:
        while True:
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5)
                parts = ["ui=up"]
            except Exception:
                parts = ["ui=DOWN"]
            for n in names:
                if n == "cloudflared" and not os.path.exists(_pidfile(n)):
                    parts.append("tunnel=off")
                else:
                    parts.append("%s=%s" % (n, "up" if process_alive(n) else "DOWN"))
            gpus = gpu_info()
            if gpus:
                parts.append("GPU " + ", ".join("%d: %.1f/%.0f GB %d%%" % (g["index"], g["mem_used_mb"] / 1024,
                                                                           g["mem_total_mb"] / 1024, g["util"])
                                                for g in gpus))
            if extra:
                try:
                    s = extra()
                    if s:
                        parts.append(s)
                except Exception as e:
                    parts.append("status error: %s" % e)
            print(time.strftime("%H:%M:%S"), " ".join(parts), flush=True)
            time.sleep(interval)
    except KeyboardInterrupt:
        print("Stopped watching. Everything keeps running in the background; re-run this cell to watch again.")


def api_get(port, path, password, timeout=10):
    """GET a JSON endpoint of a local app (used by keep-alive status lines)."""
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (port, path),
                                 headers={"Authorization": "Bearer %s" % password} if password else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))
''')
print("wrote", os.path.join(APP_DIR, 'studio_http.py'))

In [ ]:
# Shared page styles
# Writes kit.css into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.css'), "w", encoding="utf-8") as f:
    f.write(r'''/* Shared look for the agentic-notebooks app pages (served at /static/kit.css). */
:root{--bg:#f5f6f8;--panel:#fff;--panel2:#f0f2f5;--text:#1b1f24;--muted:#626b76;--line:#dde1e6;
  --accent:#2f6fec;--accent-ink:#fff;--ok:#1a7f4b;--warn:#a96500;--bad:#c4372f;--radius:10px;
  --mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0f1114;--panel:#171a1f;--panel2:#1e2228;--text:#e7e9ec;
  --muted:#9aa3ad;--line:#2b3038;--accent:#5b8ff5;--ok:#3fbf7f;--warn:#e0a03a;--bad:#ef6a60}}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
a{color:var(--accent)}
header.top{display:flex;align-items:center;gap:16px;flex-wrap:wrap;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel)}
header.top h1{font-size:17px;margin:0;font-weight:650}
header.top .sub{color:var(--muted);font-size:13px}
header.top .right{margin-left:auto;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
nav.tabs{display:flex;gap:4px;padding:0 20px;border-bottom:1px solid var(--line);background:var(--panel);overflow-x:auto}
nav.tabs button{border:0;background:none;color:var(--muted);padding:10px 12px;font:inherit;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap}
nav.tabs button.on{color:var(--text);border-bottom-color:var(--accent);font-weight:600}
main{max-width:1180px;margin:0 auto;padding:20px}
[data-pane]{display:none}[data-pane].on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:var(--radius);padding:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 12px}
.card h3{font-size:14px;margin:14px 0 8px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.col{display:flex;flex-direction:column;gap:6px}
.muted{color:var(--muted)}.small{font-size:12px}.mono{font-family:var(--mono);font-size:12.5px}
label{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:var(--muted)}
label.inline{flex-direction:row;align-items:center;gap:6px;color:var(--text);font-size:13px}
input,select,textarea{font:inherit;color:var(--text);background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
input[type=checkbox]{width:auto}
textarea{width:100%;resize:vertical;min-height:90px}
input:focus,select:focus,textarea:focus{outline:2px solid color-mix(in srgb,var(--accent) 40%,transparent);border-color:var(--accent)}
button,.btn{font:inherit;border:1px solid var(--line);background:var(--panel2);color:var(--text);border-radius:8px;padding:7px 13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}
button.primary,.btn.primary{background:var(--accent);border-color:var(--accent);color:var(--accent-ink);font-weight:600}
button.danger{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
table{width:100%;border-collapse:collapse}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{font-size:12px;color:var(--muted);font-weight:600}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;background:var(--panel2);border:1px solid var(--line)}
.badge.ok{color:var(--ok)}.badge.warn{color:var(--warn)}.badge.bad{color:var(--bad)}
.bar{height:6px;background:var(--panel2);border-radius:99px;overflow:hidden}
.bar>i{display:block;height:100%;background:var(--accent);transition:width .3s}
pre.log{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:10px;max-height:320px;overflow:auto;font:12px/1.45 var(--mono);white-space:pre-wrap;margin:0}
.drop{border:2px dashed var(--line);border-radius:var(--radius);padding:22px;text-align:center;color:var(--muted)}
.drop.over{border-color:var(--accent);color:var(--text)}
#toasts{position:fixed;right:16px;bottom:16px;display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--panel);border:1px solid var(--line);border-left:4px solid var(--accent);padding:10px 14px;border-radius:8px;box-shadow:0 6px 20px rgba(0,0,0,.15);max-width:380px}
.toast.bad{border-left-color:var(--bad)}.toast.ok{border-left-color:var(--ok)}
.empty{color:var(--muted);padding:18px;text-align:center}
@media (max-width:640px){main{padding:12px}header.top{padding:10px 12px}nav.tabs{padding:0 8px}}
''')
print("wrote", os.path.join(APP_DIR, 'kit.css'))

In [ ]:
# Shared page scripts
# Writes kit.js into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.js'), "w", encoding="utf-8") as f:
    f.write(r'''// Shared helpers for the agentic-notebooks app pages (served at /static/kit.js).
// The browser already holds the HTTP Basic login, so fetch() calls need no token.
const $ = (sel, root = document) => root.querySelector(sel);
const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

function el(tag, attrs = {}, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null || v === false) continue;
    if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (k === "html") e.innerHTML = v;
    else if (k === "style" && typeof v === "object") Object.assign(e.style, v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const kid of kids.flat()) if (kid != null && kid !== false) e.append(kid.nodeType ? kid : String(kid));
  return e;
}

async function api(path, opts = {}) {
  const init = { method: opts.method || (opts.json !== undefined || opts.body ? "POST" : "GET"), headers: {} };
  if (opts.json !== undefined) { init.body = JSON.stringify(opts.json); init.headers["Content-Type"] = "application/json"; }
  else if (opts.body) init.body = opts.body;
  const r = await fetch(path, init);
  if (r.status === 401) { location.reload(); throw new Error("Signed out; please sign in again"); }
  const ct = r.headers.get("Content-Type") || "";
  const data = ct.includes("json") ? await r.json() : await r.text();
  if (!r.ok) throw new Error((data && data.error) || r.status + " " + r.statusText);
  return data;
}

function toast(msg, kind = "") {
  let box = $("#toasts");
  if (!box) { box = el("div", { id: "toasts" }); document.body.append(box); }
  const t = el("div", { class: "toast " + kind }, msg);
  box.append(t);
  setTimeout(() => t.remove(), kind === "bad" ? 9000 : 4500);
}

async function guard(fn, okMsg) {
  try { const r = await fn(); if (okMsg) toast(okMsg, "ok"); return r; }
  catch (e) { toast(e.message || String(e), "bad"); throw e; }
}

function setupTabs(onChange) {
  const btns = $$("nav.tabs button[data-tab]");
  const show = (name) => {
    btns.forEach(b => b.classList.toggle("on", b.dataset.tab === name));
    $$("[data-pane]").forEach(p => p.classList.toggle("on", p.dataset.pane === name));
    try { history.replaceState(null, "", "#" + name); } catch (e) {}
    if (onChange) onChange(name);
  };
  btns.forEach(b => b.addEventListener("click", () => show(b.dataset.tab)));
  const first = location.hash.slice(1);
  show(btns.some(b => b.dataset.tab === first) ? first : btns[0].dataset.tab);
  return show;
}

function fmtBytes(n) {
  if (n == null) return "";
  const u = ["B", "KB", "MB", "GB", "TB"]; let i = 0;
  while (n >= 1024 && i < u.length - 1) { n /= 1024; i++; }
  return (i ? n.toFixed(1) : n) + " " + u[i];
}

function fmtDur(s) {
  if (s == null || isNaN(s)) return "";
  s = Math.max(0, Math.round(s));
  const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), sec = s % 60;
  return (h ? h + ":" + String(m).padStart(2, "0") : m) + ":" + String(sec).padStart(2, "0");
}

function fmtAgo(ts) {
  if (!ts) return "";
  const d = Date.now() / 1000 - ts;
  if (d < 60) return "just now";
  if (d < 3600) return Math.floor(d / 60) + " min ago";
  if (d < 86400) return Math.floor(d / 3600) + " h ago";
  return new Date(ts * 1000).toLocaleString();
}

function badge(state) {
  const kind = { done: "ok", ready: "ok", running: "warn", queued: "", loading: "warn", error: "bad", stopped: "bad" }[state] || "";
  return el("span", { class: "badge " + kind }, state || "?");
}

function progressBar(frac) {
  return el("div", { class: "bar" }, el("i", { style: { width: Math.round(Math.max(0, Math.min(1, frac || 0)) * 100) + "%" } }));
}

function every(ms, fn) {
  let stopped = false;
  const tick = async () => { if (stopped) return; try { await fn(); } catch (e) {} if (!stopped) setTimeout(tick, ms); };
  tick();
  return () => { stopped = true; };
}

function dropZone(zone, input, onFiles) {
  zone.addEventListener("click", () => input.click());
  input.addEventListener("change", () => { if (input.files.length) onFiles([...input.files]); input.value = ""; });
  zone.addEventListener("dragover", e => { e.preventDefault(); zone.classList.add("over"); });
  zone.addEventListener("dragleave", () => zone.classList.remove("over"));
  zone.addEventListener("drop", e => { e.preventDefault(); zone.classList.remove("over"); if (e.dataTransfer.files.length) onFiles([...e.dataTransfer.files]); });
}

function mcpHelp(name) {
  const url = location.origin + "/mcp";
  return "claude mcp add --transport http " + name + " " + url + " --header \"Authorization: Bearer <password>\"";
}
''')
print("wrote", os.path.join(APP_DIR, 'kit.js'))

In [ ]:
# Dataset formats and statistics
# Writes lab_data.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'lab_data.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Dataset handling for the Unsloth Fine-tuning Lab (pure Python, no GPU libraries).

Every record is normalised to one of two shapes:
    {"messages": [{"role": "system"|"user"|"assistant", "content": "..."}, ...]}   chat / instruction data
    {"text": "..."}                                                                raw text (continued pre-training)
Accepted inputs: OpenAI-style messages, ShareGPT conversations (from/value), Alpaca
(instruction/input/output), prompt/completion, question/answer, input/output, query/response, text.
"""
import csv
import io
import json

ROLES = {"system": "system", "user": "user", "human": "user", "assistant": "assistant", "gpt": "assistant",
         "bot": "assistant", "model": "assistant", "tool": "tool"}
PAIRS = (("prompt", "completion"), ("question", "answer"), ("input", "output"), ("query", "response"),
         ("instruction", "response"))


def _content(c):
    if isinstance(c, list):        # OpenAI content parts
        c = "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)
    if not isinstance(c, str):
        raise ValueError("message content must be text")
    return c


def normalize(obj):
    if not isinstance(obj, dict):
        raise ValueError("each record must be a JSON object")
    msgs = obj.get("messages", obj.get("conversations", obj.get("conversation")))
    if msgs is not None:
        if not isinstance(msgs, list) or not msgs:
            raise ValueError("messages must be a non-empty list")
        out = []
        for m in msgs:
            if not isinstance(m, dict):
                raise ValueError("each message must be an object")
            role = ROLES.get(str(m.get("role", m.get("from", ""))).lower())
            if not role:
                raise ValueError("unknown role %r" % m.get("role", m.get("from")))
            out.append({"role": role, "content": _content(m.get("content", m.get("value", "")))})
        if not any(m["role"] == "assistant" and m["content"].strip() for m in out):
            raise ValueError("a conversation needs at least one non-empty assistant message")
        return {"messages": out}
    system = obj.get("system") or obj.get("system_prompt")
    head = [{"role": "system", "content": _content(system)}] if system else []
    if "instruction" in obj and "output" in obj:
        if not _content(obj["output"]).strip() or not _content(obj["instruction"]).strip():
            raise ValueError("instruction and output must not be empty")
        user = _content(obj["instruction"]) + ("\n\n" + _content(obj["input"]) if obj.get("input") else "")
        return {"messages": head + [{"role": "user", "content": user},
                                    {"role": "assistant", "content": _content(obj["output"])}]}
    for q, a in PAIRS:
        if q in obj and a in obj:
            if not _content(obj[a]).strip():
                raise ValueError("%s is empty" % a)
            return {"messages": head + [{"role": "user", "content": _content(obj[q])},
                                        {"role": "assistant", "content": _content(obj[a])}]}
    if isinstance(obj.get("text"), str) and obj["text"].strip():
        return {"text": obj["text"]}
    raise ValueError("unrecognised record; use messages, conversations, instruction/output, prompt/completion, "
                     "question/answer or text (keys found: %s)" % ", ".join(sorted(obj)[:8]))


def parse(text, fmt="auto"):
    """Returns (records, errors) where errors is a list of {line, error}."""
    text = text.lstrip("﻿")
    if fmt == "auto":
        s = text.lstrip()
        fmt = "json" if s.startswith("[") else "jsonl" if s.startswith("{") else "csv"
    recs, errs = [], []
    if fmt == "json":
        try:
            items = json.loads(text)
        except ValueError as e:
            return [], [{"line": 0, "error": "not valid JSON: %s" % e}]
        if not isinstance(items, list):
            return [], [{"line": 0, "error": "a .json file must hold a list of records"}]
        rows = list(enumerate(items, 1))
    elif fmt == "jsonl":
        rows = []
        for n, line in enumerate(text.splitlines(), 1):
            if line.strip():
                try:
                    rows.append((n, json.loads(line)))
                except ValueError as e:
                    errs.append({"line": n, "error": "not valid JSON: %s" % e})
    elif fmt == "csv":
        rows = [(n, {k.strip().lower(): v for k, v in r.items() if k}) for n, r in
                enumerate(csv.DictReader(io.StringIO(text)), 2)]
    else:
        raise ValueError("format must be jsonl, json, csv or auto")
    for n, obj in rows:
        try:
            recs.append(normalize(obj))
        except ValueError as e:
            errs.append({"line": n, "error": str(e)})
    return recs, errs


def preview(rec, n=160):
    if "text" in rec:
        s = rec["text"]
    else:
        user = next((m["content"] for m in rec["messages"] if m["role"] == "user"), "")
        bot = next((m["content"] for m in rec["messages"] if m["role"] == "assistant"), "")
        s = "%s  →  %s" % (user, bot)
    s = " ".join(s.split())
    return s[:n] + ("…" if len(s) > n else "")


def chars(rec):
    return len(rec["text"]) if "text" in rec else sum(len(m["content"]) for m in rec["messages"])


def stats(records):
    if not records:
        return {"count": 0}
    lens = sorted(chars(r) for r in records)
    chat = sum(1 for r in records if "messages" in r)
    return {"count": len(records), "chat": chat, "text": len(records) - chat,
            "turns_avg": round(sum(len(r["messages"]) for r in records if "messages" in r) / chat, 1) if chat else 0,
            "chars_median": lens[len(lens) // 2], "chars_max": lens[-1],
            "approx_tokens_total": sum(lens) // 4, "approx_tokens_max": lens[-1] // 4}
''')
print("wrote", os.path.join(APP_DIR, 'lab_data.py'))

In [ ]:
# Training process (Unsloth QLoRA + TRL SFTTrainer)
# Writes train_unsloth.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'train_unsloth.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Fine-tunes a model with Unsloth (4-bit QLoRA). Launched by lab_server.py as its own process:
    python train_unsloth.py <run_dir>/job.json
It writes <run_dir>/status.json while it runs, the LoRA adapter to <run_dir>/adapter, and
metrics.json + samples.json at the end. SIGTERM stops training early; the adapter trained so far
is still saved.
"""
import json
import math
import os
import random
import signal
import sys
import time
import traceback

JOB = json.load(open(sys.argv[1], encoding="utf-8"))
RD = JOB["run_dir"]
ST = {"phase": "starting", "started": time.time()}
STOP = {"flag": False}
LOSSES, EVALS = [], []


def status(**kw):
    ST.update(kw, updated=time.time())
    tmp = os.path.join(RD, "status.json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(ST, f)
    os.replace(tmp, os.path.join(RD, "status.json"))


def on_term(*_):
    STOP["flag"] = True
    print("Stop requested: finishing the current step, then saving the adapter.", flush=True)


signal.signal(signal.SIGTERM, on_term)


def sft_config(SFTConfig, n_train, n_eval):
    """Builds SFTConfig with whichever argument names this TRL version uses."""
    fields = set(getattr(SFTConfig, "__dataclass_fields__", {}))
    bs, ga = JOB["batch_size"], JOB["grad_accum"]
    steps_per_epoch = max(1, math.ceil(n_train / (bs * ga)))
    total = JOB["max_steps"] if JOB["max_steps"] > 0 else math.ceil(steps_per_epoch * JOB["epochs"])
    import torch
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    cfg = dict(output_dir=os.path.join(RD, "checkpoints"), per_device_train_batch_size=bs,
               gradient_accumulation_steps=ga, learning_rate=JOB["learning_rate"], num_train_epochs=JOB["epochs"],
               max_steps=JOB["max_steps"] if JOB["max_steps"] > 0 else -1, warmup_ratio=0.03,
               lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01, logging_steps=1,
               save_strategy="no", report_to="none", seed=JOB["seed"], fp16=not bf16, bf16=bf16,
               dataset_text_field="text", packing=False, dataset_num_proc=2)
    cfg["max_seq_length" if "max_seq_length" in fields else "max_length"] = JOB["max_seq_length"]
    if n_eval:
        cfg["eval_strategy" if "eval_strategy" in fields else "evaluation_strategy"] = "steps"
        cfg["eval_steps"] = max(1, total // 5)
        cfg["per_device_eval_batch_size"] = bs
    dropped = sorted(k for k in cfg if fields and k not in fields)
    if dropped:
        print("SFTConfig in this TRL version has no", dropped, flush=True)
    return SFTConfig(**{k: v for k, v in cfg.items() if not fields or k in fields}), total


def main():
    status(phase="loading model", base=JOB["base"])
    from unsloth import FastLanguageModel           # import unsloth first: it patches transformers and trl
    from unsloth.chat_templates import get_chat_template
    import inspect
    import torch
    from datasets import Dataset
    from transformers import TrainerCallback
    from trl import SFTConfig, SFTTrainer

    model, tok = FastLanguageModel.from_pretrained(model_name=JOB["base"], max_seq_length=JOB["max_seq_length"],
                                                   load_in_4bit=True, dtype=None, token=os.environ.get("HF_TOKEN"))
    if getattr(tok, "chat_template", None) is None:
        tok = get_chat_template(tok, chat_template="chatml")
    model = FastLanguageModel.get_peft_model(
        model, r=JOB["lora_r"], lora_alpha=JOB["lora_alpha"], lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=JOB["seed"])

    status(phase="preparing data")
    recs = [json.loads(line) for line in open(JOB["dataset"], encoding="utf-8") if line.strip()]
    eos = tok.eos_token or ""
    texts = [tok.apply_chat_template(r["messages"], tokenize=False) if "messages" in r else r["text"] + eos
             for r in recs]
    order = list(range(len(texts)))
    random.Random(JOB["seed"]).shuffle(order)
    n_eval = min(200, int(len(texts) * JOB["val_frac"])) if len(texts) >= 20 else 0
    eval_idx, train_idx = order[:n_eval], order[n_eval:]
    train_ds = Dataset.from_dict({"text": [texts[i] for i in train_idx]})
    eval_ds = Dataset.from_dict({"text": [texts[i] for i in eval_idx]}) if n_eval else None
    args, total = sft_config(SFTConfig, len(train_idx), n_eval)

    t0 = time.time()

    class Progress(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kw):
            logs = logs or {}
            if "loss" in logs:
                LOSSES.append([state.global_step, round(logs["loss"], 4)])
            if "eval_loss" in logs:
                EVALS.append([state.global_step, round(logs["eval_loss"], 4)])
            done = max(1, state.global_step)
            el = time.time() - t0
            status(phase="training", step=state.global_step, total_steps=state.max_steps or total,
                   epoch=round(state.epoch or 0, 3), loss=LOSSES[-1][1] if LOSSES else None,
                   eval_loss=EVALS[-1][1] if EVALS else None, lr=logs.get("learning_rate", ST.get("lr")),
                   losses=LOSSES[-1000:], evals=EVALS, elapsed=round(el),
                   eta=round(el / done * max(0, (state.max_steps or total) - state.global_step)))

        def on_step_end(self, args, state, control, **kw):
            if STOP["flag"]:
                control.should_training_stop = True
            return control

    params = inspect.signature(SFTTrainer.__init__).parameters
    kw = dict(model=model, train_dataset=train_ds, eval_dataset=eval_ds, args=args, callbacks=[Progress()])
    kw["processing_class" if "processing_class" in params else "tokenizer"] = tok
    trainer = SFTTrainer(**kw)
    status(phase="training", step=0, total_steps=total, n_train=len(train_idx), n_eval=n_eval)
    result = trainer.train()

    status(phase="saving adapter")
    adapter = os.path.join(RD, "adapter")
    model.save_pretrained(adapter)
    tok.save_pretrained(adapter)
    metrics = {"train_loss": round(result.training_loss, 4) if result.training_loss else None,
               "final_loss": LOSSES[-1][1] if LOSSES else None,
               "eval_loss": EVALS[-1][1] if EVALS else None, "first_eval_loss": EVALS[0][1] if EVALS else None,
               "steps": trainer.state.global_step, "planned_steps": total, "stopped_early": STOP["flag"],
               "seconds": round(time.time() - t0), "n_train": len(train_idx), "n_eval": n_eval,
               "peak_gpu_gb": round(torch.cuda.max_memory_reserved() / 2 ** 30, 2) if torch.cuda.is_available() else None}
    with open(os.path.join(RD, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=1)

    samples = []
    held_out = [recs[i] for i in eval_idx if "messages" in recs[i]][:3]
    if held_out and not STOP["flag"]:
        status(phase="writing sample answers")
        FastLanguageModel.for_inference(model)
        for r in held_out:
            msgs = r["messages"]
            last = max(i for i, m in enumerate(msgs) if m["role"] == "assistant")
            prompt = msgs[:last]
            ids = tok.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="pt",
                                        return_dict=True)["input_ids"].to(model.device)
            out = model.generate(input_ids=ids, max_new_tokens=256, do_sample=False)
            samples.append({"prompt": prompt[-1]["content"], "expected": msgs[last]["content"],
                            "model": tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()})
    with open(os.path.join(RD, "samples.json"), "w", encoding="utf-8") as f:
        json.dump(samples, f, indent=1, ensure_ascii=False)
    status(phase="stopped" if STOP["flag"] else "done", metrics=metrics)


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        traceback.print_exc()
        status(phase="error", error="%s: %s" % (type(e).__name__, e))
        sys.exit(1)
''')
print("wrote", os.path.join(APP_DIR, 'train_unsloth.py'))

In [ ]:
# Chat process
# Writes infer_worker.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'infer_worker.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Chat with a base model or a fine-tuned run. Started by lab_server.py as its own process:
    MODEL_PATH=<hub id or run adapter dir> MODEL_KEY=<label> python infer_worker.py --port 7861
Listens on 127.0.0.1 only. GET /health -> {"state": "loading"|"ready"|"error", "model": key};
POST /generate {"messages": [...], "max_new_tokens", "temperature", "top_p"} -> {"text", "tokens", "seconds"}.
"""
import argparse
import json
import os
import threading
import time
import traceback
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

STATE = {"state": "loading", "model": os.environ.get("MODEL_KEY", os.environ.get("MODEL_PATH")), "error": None}
M = {}
GEN_LOCK = threading.Lock()


def load():
    try:
        from unsloth import FastLanguageModel
        from unsloth.chat_templates import get_chat_template
        model, tok = FastLanguageModel.from_pretrained(model_name=os.environ["MODEL_PATH"],
                                                       max_seq_length=int(os.environ.get("MAX_SEQ", "4096")),
                                                       load_in_4bit=True, dtype=None, token=os.environ.get("HF_TOKEN"))
        if getattr(tok, "chat_template", None) is None:
            tok = get_chat_template(tok, chat_template="chatml")
        FastLanguageModel.for_inference(model)
        M.update(model=model, tok=tok)
        STATE.update(state="ready", loaded=time.time())
        print("ready:", STATE["model"], flush=True)
    except Exception as e:
        traceback.print_exc()
        STATE.update(state="error", error="%s: %s" % (type(e).__name__, str(e)[:500]))


def generate(b):
    msgs = b.get("messages")
    if not isinstance(msgs, list) or not msgs:
        raise ValueError("messages must be a non-empty list")
    tok, model = M["tok"], M["model"]
    max_new = max(1, min(2048, int(b.get("max_new_tokens") or 512)))
    temp = float(b.get("temperature", 0.7))
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                return_dict=True)["input_ids"].to(model.device)
    t = time.time()
    with GEN_LOCK:
        kw = dict(input_ids=ids, max_new_tokens=max_new, do_sample=temp > 0)
        if temp > 0:
            kw.update(temperature=temp, top_p=float(b.get("top_p", 0.9)))
        out = model.generate(**kw)
    new = out[0][ids.shape[1]:]
    return {"text": tok.decode(new, skip_special_tokens=True).strip(), "tokens": int(new.shape[0]),
            "prompt_tokens": int(ids.shape[1]), "seconds": round(time.time() - t, 2), "model": STATE["model"]}


class H(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *a):
        pass

    def send(self, code, obj):
        data = json.dumps(obj, ensure_ascii=False).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        self.send(200, STATE) if self.path == "/health" else self.send(404, {"error": "not found"})

    def do_POST(self):
        body = self.rfile.read(int(self.headers.get("Content-Length") or 0))
        if self.path != "/generate":
            return self.send(404, {"error": "not found"})
        if STATE["state"] != "ready":
            return self.send(409, {"error": "model is %s" % STATE["state"], **STATE})
        try:
            self.send(200, generate(json.loads(body or b"{}")))
        except ValueError as e:
            self.send(400, {"error": str(e)})
        except Exception as e:
            traceback.print_exc()
            self.send(500, {"error": "%s: %s" % (type(e).__name__, e)})


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7861)
    srv = ThreadingHTTPServer(("127.0.0.1", ap.parse_args().port), H)
    threading.Thread(target=load, daemon=True).start()
    srv.serve_forever()
''')
print("wrote", os.path.join(APP_DIR, 'infer_worker.py'))

In [ ]:
# Export process (merged 16-bit, GGUF)
# Writes export_unsloth.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'export_unsloth.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Exports a fine-tuned run. Launched by lab_server.py as its own process:
    python export_unsloth.py <export_dir>/job.json
Formats: merged_16bit (a full Hugging Face model), gguf_q4_k_m / gguf_q8_0 / gguf_f16 (for llama.cpp,
Ollama, LM Studio). GGUF builds llama.cpp on first use, which takes several minutes.
Writes <export_dir>/status.json; the exported files end up in <export_dir>/model.
"""
import glob
import json
import os
import shutil
import sys
import time
import traceback

JOB = json.load(open(sys.argv[1], encoding="utf-8"))
OUT = JOB["export_dir"]


def status(**kw):
    st = json.load(open(os.path.join(OUT, "status.json"))) if os.path.exists(os.path.join(OUT, "status.json")) else {}
    st.update(kw, updated=time.time())
    with open(os.path.join(OUT, "status.json.tmp"), "w") as f:
        json.dump(st, f)
    os.replace(os.path.join(OUT, "status.json.tmp"), os.path.join(OUT, "status.json"))


def main():
    status(phase="loading model", started=time.time())
    from unsloth import FastLanguageModel
    model, tok = FastLanguageModel.from_pretrained(model_name=JOB["adapter"], max_seq_length=JOB["max_seq_length"],
                                                   load_in_4bit=True, dtype=None, token=os.environ.get("HF_TOKEN"))
    dest = os.path.join(OUT, "model")
    fmt = JOB["format"]
    if fmt == "merged_16bit":
        status(phase="merging LoRA into 16-bit weights")
        model.save_pretrained_merged(dest, tok, save_method="merged_16bit")
    elif fmt.startswith("gguf_"):
        status(phase="converting to GGUF %s (builds llama.cpp the first time)" % fmt[5:])
        model.save_pretrained_gguf(dest, tok, quantization_method=fmt[5:])
        # some Unsloth versions write next to the folder (dest + "_gguf") instead of into it
        os.makedirs(dest, exist_ok=True)
        for f in glob.glob(dest + "_gguf/*.gguf") + glob.glob(os.path.join(os.path.dirname(dest), "*.gguf")):
            shutil.move(f, os.path.join(dest, os.path.basename(f)))
        if not glob.glob(os.path.join(dest, "*.gguf")):
            raise RuntimeError("no .gguf file was produced; see the export log")
    else:
        raise ValueError("unknown format %s" % fmt)
    size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(dest) for f in fn)
    status(phase="done", size=size, files=sorted(os.listdir(dest)))


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        traceback.print_exc()
        status(phase="error", error="%s: %s" % (type(e).__name__, str(e)[:800]))
        sys.exit(1)
''')
print("wrote", os.path.join(APP_DIR, 'export_unsloth.py'))

In [ ]:
# Lab server on port 7860: API and MCP
# Writes lab_server.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'lab_server.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Unsloth Fine-tuning Lab: the minimalist LLM factory on port 7860.

Runs as its own process (started by the notebook with studio_http.spawn):
    APP_PASSWORD=... HF_TOKEN=... python lab_server.py --port 7860

Dataset in (upload, Hugging Face Hub, or added by an agent) -> 4-bit LoRA fine-tune with Unsloth ->
chat with the result -> export (LoRA adapter, merged 16-bit, GGUF) -> download or push to the Hub.

GPU plan with two GPUs: training on GPU 0, chat and exports on GPU 1. With one GPU, chat and exports
wait until training is finished. Training, chat and exports each run in their own process, so a crash
or an out-of-memory error never takes the server down, and they keep running if the server restarts.
"""
import argparse
import json
import os
import re
import shutil
import signal
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request

import lab_data as D
import studio_http as K

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("WORK_DIR", "/kaggle/working/unsloth_lab")
EXPORTS = os.environ.get("EXPORT_DIR", "/tmp/unsloth_lab/exports")     # big files: outside /kaggle/working's 20 GB
LOGS = os.environ.get("LOG_DIR", "/kaggle/working/logs")
HF_TOKEN = os.environ.get("HF_TOKEN") or None
INFER_PORT = int(os.environ.get("INFER_PORT", "7861"))
MAX_WAIT = 85
BASE_MODELS = [
    {"id": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit", "label": "Llama 3.2 1B Instruct (fastest)"},
    {"id": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", "label": "Llama 3.2 3B Instruct"},
    {"id": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "label": "Qwen2.5 1.5B Instruct"},
    {"id": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", "label": "Qwen2.5 3B Instruct"},
    {"id": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "label": "Qwen2.5 7B Instruct (T4: short sequences)"},
    {"id": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", "label": "Llama 3.1 8B Instruct (T4: short sequences)"},
    {"id": "unsloth/gemma-2-2b-it-bnb-4bit", "label": "Gemma 2 2B it"},
    {"id": "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", "label": "Mistral 7B Instruct v0.3"},
]
EXPORT_FORMATS = ["merged_16bit", "gguf_q4_k_m", "gguf_q8_0", "gguf_f16"]
HUB_ID = re.compile(r"^[A-Za-z0-9][\w.-]*/[\w.-]+$")
log = K.file_logger(os.environ.get("APP_LOG", os.path.join(LOGS, "unsloth_lab.log")), "lab")

LOCK = threading.RLock()
DATA = {"records": []}
IMPORT = {"state": "idle"}
PUSH = {}
INFER = {"key": None, "since": 0}


# ====================================================================== storage
def p(*parts):
    return os.path.join(WORK, *parts)


def load_dataset():
    recs = []
    if os.path.exists(p("dataset.jsonl")):
        recs, errs = D.parse(open(p("dataset.jsonl"), encoding="utf-8").read(), "jsonl")
        if errs:
            log.warning("dataset.jsonl: skipped %d bad lines", len(errs))
    DATA["records"] = recs


def save_dataset():
    tmp = p("dataset.jsonl.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in DATA["records"]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, p("dataset.jsonl"))


def add_records(recs, mode="append"):
    if mode not in ("append", "replace"):
        raise ValueError("mode must be append or replace")
    with LOCK:
        DATA["records"] = list(recs) if mode == "replace" else DATA["records"] + list(recs)
        save_dataset()
        return len(DATA["records"])


def dataset_info():
    with LOCK:
        return dict(D.stats(DATA["records"]), import_job=dict(IMPORT))


def import_hf(name, split="train", config=None, max_rows=5000, mode="append"):
    if not HUB_ID.match(str(name or "")):
        raise ValueError("name must look like owner/dataset")
    max_rows = max(1, min(200000, int(max_rows or 5000)))
    if IMPORT.get("state") == "running":
        raise ValueError("an import is already running")
    IMPORT.clear()
    IMPORT.update(state="running", name=name, split=split, started=time.time())

    def go():
        try:
            from datasets import load_dataset
            ds = load_dataset(name, config or None, split="%s[:%d]" % (split, max_rows), token=HF_TOKEN)
            recs, errs = [], 0
            for row in ds:
                try:
                    recs.append(D.normalize(dict(row)))
                except ValueError:
                    errs += 1
            if not recs:
                raise ValueError("no usable rows (columns: %s)" % ", ".join(ds.column_names))
            total = add_records(recs, mode)
            IMPORT.update(state="done", added=len(recs), skipped=errs, total=total)
            log.info("imported %d rows from %s", len(recs), name)
        except Exception as e:
            log.error("import failed\n%s", traceback.format_exc())
            IMPORT.update(state="error", error=str(e)[:500])
    threading.Thread(target=go, daemon=True).start()
    return dict(IMPORT)


# ====================================================================== GPUs
def gpu_count():
    return len(K.gpu_info())


def training_running():
    return K.process_alive("lab-train")


def side_gpu():
    """GPU for chat and exports: GPU 1 with two GPUs; GPU 0 only while no training runs."""
    n = gpu_count()
    if n >= 2:
        return "1"
    if n == 1:
        if training_running():
            raise ValueError("training is using the only GPU; chat and exports are available when it finishes "
                             "(or pick the 'GPU T4 x2' accelerator)")
        return "0"
    raise ValueError("no GPU: pick a GPU accelerator in the notebook settings")


# ====================================================================== training
def run_dir(name):
    name = K.safe_name(name)
    if not name or not os.path.isfile(p("runs", name, "job.json")):
        raise K.HTTPError(404, "unknown run %r" % name)
    return p("runs", name)


def current_run():
    return (K.read_json(p("current_run.json"), {}) or {}).get("run")


def start_training(o):
    with LOCK:
        if training_running():
            raise ValueError("a training run is already in progress")
        if not gpu_count():
            raise ValueError("no GPU: pick a GPU accelerator in the notebook settings")
        if len(DATA["records"]) < 10:
            raise ValueError("add at least 10 examples first (you have %d)" % len(DATA["records"]))
        name = K.safe_name(o.get("name")) or time.strftime("run-%Y%m%d-%H%M%S")
        if os.path.exists(p("runs", name)):
            raise ValueError("a run named %r already exists" % name)
        base = str(o.get("base_model") or BASE_MODELS[0]["id"]).strip()
        if base.startswith("run:"):
            prev = os.path.join(run_dir(base[4:]), "adapter")
            if not os.path.exists(os.path.join(prev, "adapter_config.json")):
                raise ValueError("that run has no saved adapter")
            base = prev
        elif not HUB_ID.match(base):
            raise ValueError("base_model must be a Hugging Face id like unsloth/Llama-3.2-3B-Instruct-bnb-4bit")

        def num(key, default, cast, lo, hi):
            v = o.get(key)
            v = default if v in (None, "") else cast(v)
            if not lo <= v <= hi:
                raise ValueError("%s must be between %s and %s" % (key, lo, hi))
            return v
        rd = p("runs", name)
        os.makedirs(rd)
        shutil.copy(p("dataset.jsonl"), os.path.join(rd, "dataset.jsonl"))
        job = {"name": name, "run_dir": rd, "dataset": os.path.join(rd, "dataset.jsonl"), "base": base,
               "base_label": str(o.get("base_model") or BASE_MODELS[0]["id"]),
               "epochs": num("epochs", 2, float, 0.1, 50), "max_steps": num("max_steps", 0, int, 0, 100000),
               "learning_rate": num("learning_rate", 2e-4, float, 1e-6, 1e-2),
               "lora_r": num("lora_r", 16, int, 1, 256), "lora_alpha": num("lora_alpha", 16, int, 1, 512),
               "max_seq_length": num("max_seq_length", 2048, int, 128, 32768),
               "batch_size": num("batch_size", 2, int, 1, 64), "grad_accum": num("grad_accum", 4, int, 1, 128),
               "val_frac": num("val_frac", 0.05, float, 0.0, 0.5), "seed": num("seed", 3407, int, 0, 2 ** 31),
               "created": time.time(), "examples": len(DATA["records"])}
        K.write_json(os.path.join(rd, "job.json"), job)
        K.write_json(os.path.join(rd, "status.json"), {"phase": "starting"})
        if gpu_count() == 1:
            stop_infer()                        # free the only GPU
        env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
                   PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        K.spawn("lab-train", [sys.executable, os.path.join(HERE, "train_unsloth.py"), os.path.join(rd, "job.json")],
                os.path.join(rd, "train.log"), env=env, cwd=HERE)
        K.write_json(p("current_run.json"), {"run": name})
        log.info("training %s on %s", name, base)
        return {"run": name, "job": job}


def run_status(name):
    rd = p("runs", name)
    st = K.read_json(os.path.join(rd, "status.json"), {}) or {}
    phase = st.get("phase", "?")
    live = name == current_run() and training_running()
    if not live and phase not in ("done", "error", "stopped"):
        phase = "error"
        st.setdefault("error", "the training process exited early; see the log")
    st["phase"] = phase
    st["state"] = "running" if live else phase
    return st


def train_status(with_log=True):
    name = current_run()
    if not name or not os.path.isdir(p("runs", name)):
        return {"state": "idle"}
    st = run_status(name)
    st["run"] = name
    if with_log:
        st["log"] = K.tail(os.path.join(p("runs", name), "train.log"), 40)
    return st


def stop_training(force=False):
    info = K.read_json(os.path.join(K.PID_DIR, "lab-train.json"))
    if not info or not training_running():
        return {"stopped": False, "reason": "no training is running"}
    if force:
        K.stop_process("lab-train", timeout=5)
        return {"stopped": True, "note": "killed; the adapter was not saved"}
    os.killpg(info["pid"], signal.SIGTERM)     # train_unsloth.py finishes the step, then saves the adapter
    return {"stopped": True, "note": "stopping after the current step; the adapter trained so far will be saved"}


def list_runs():
    out = []
    if not os.path.isdir(p("runs")):
        return out
    for name in sorted(os.listdir(p("runs")), reverse=True):
        rd = p("runs", name)
        job = K.read_json(os.path.join(rd, "job.json"))
        if not job:
            continue
        st = run_status(name)
        out.append({"name": name, "base": job.get("base_label"), "state": st["state"], "created": job["created"],
                    "examples": job.get("examples"), "metrics": K.read_json(os.path.join(rd, "metrics.json")),
                    "has_adapter": os.path.exists(os.path.join(rd, "adapter", "adapter_config.json")),
                    "exports": export_list(name), "push": PUSH.get(name),
                    "params": {k: job[k] for k in ("epochs", "max_steps", "learning_rate", "lora_r", "lora_alpha",
                                                   "max_seq_length", "batch_size", "grad_accum")}})
    return out


def run_detail(name):
    rd = run_dir(name)
    return {"name": name, "job": K.read_json(os.path.join(rd, "job.json")), "status": run_status(name),
            "metrics": K.read_json(os.path.join(rd, "metrics.json")),
            "samples": K.read_json(os.path.join(rd, "samples.json"), []), "exports": export_list(name)}


def delete_run(name):
    rd = run_dir(name)
    if name == current_run() and training_running():
        raise ValueError("stop the run first")
    if INFER["key"] == "run:" + name:
        stop_infer()
    shutil.rmtree(rd)
    shutil.rmtree(os.path.join(EXPORTS, K.safe_name(name)), ignore_errors=True)
    return {"deleted": name}


# ====================================================================== chat (inference worker)
def infer_health():
    try:
        with urllib.request.urlopen("http://127.0.0.1:%d/health" % INFER_PORT, timeout=2) as r:
            return json.loads(r.read())
    except Exception:
        return None


def stop_infer():
    K.stop_process("lab-infer")
    INFER.update(key=None, since=0)


def model_path(key):
    if key.startswith("run:"):
        path = os.path.join(run_dir(key[4:]), "adapter")
        if not os.path.exists(os.path.join(path, "adapter_config.json")):
            raise ValueError("that run has no saved adapter yet")
        return path
    if not HUB_ID.match(key):
        raise ValueError("model must be a Hugging Face id or run:<name>")
    return key


def chat(model, messages, max_new_tokens=512, temperature=0.7):
    key = str(model or "").strip() or BASE_MODELS[0]["id"]
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    msgs = D.normalize({"messages": list(messages) + [{"role": "assistant", "content": "x"}]})["messages"][:-1]
    h = infer_health()
    if h and h.get("model") == key:
        if h["state"] == "ready":
            body = json.dumps({"messages": msgs, "max_new_tokens": min(int(max_new_tokens or 512), 1024),
                               "temperature": float(temperature)}).encode()
            req = urllib.request.Request("http://127.0.0.1:%d/generate" % INFER_PORT, data=body,
                                         headers={"Content-Type": "application/json"})
            try:
                with urllib.request.urlopen(req, timeout=MAX_WAIT) as r:
                    return dict(json.loads(r.read()), loading=False)
            except urllib.error.HTTPError as e:
                raise ValueError(json.loads(e.read() or b"{}").get("error", "generation failed"))
            except OSError:
                raise ValueError("generation took longer than %d s; ask for fewer max_new_tokens" % MAX_WAIT)
        if h["state"] == "error":
            stop_infer()
            raise ValueError("model failed to load: %s" % h.get("error"))
        return {"loading": True, "model": key, "message": "loading the model; ask again in a minute"}
    if INFER["key"] == key and time.time() - INFER["since"] < 600 and K.process_alive("lab-infer"):
        return {"loading": True, "model": key, "message": "starting the model; ask again in a minute"}
    path = model_path(key)
    gpu = side_gpu()
    with LOCK:
        if K.process_alive("lab-export"):
            raise ValueError("an export is using the GPU; chat is available when it finishes")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, MODEL_PATH=path, MODEL_KEY=key, PYTHONUNBUFFERED="1",
                   TOKENIZERS_PARALLELISM="false")
        K.spawn("lab-infer", [sys.executable, os.path.join(HERE, "infer_worker.py"), "--port", str(INFER_PORT)],
                os.path.join(LOGS, "unsloth_infer.log"), env=env, cwd=HERE)
        INFER.update(key=key, since=time.time())
    log.info("loading %s for chat on GPU %s", key, gpu)
    return {"loading": True, "model": key, "message": "loading the model (1-3 minutes); ask again then"}


def chat_status():
    h = infer_health()
    if h:
        return h
    if INFER["key"] and K.process_alive("lab-infer"):
        return {"state": "loading", "model": INFER["key"]}
    return {"state": "off"}


# ====================================================================== exports and Hub
def export_dir(name, fmt):
    return os.path.join(EXPORTS, K.safe_name(name), fmt)


def export_list(name):
    out = {}
    for fmt in EXPORT_FORMATS:
        st = K.read_json(os.path.join(export_dir(name, fmt), "status.json"))
        if st:
            if st.get("phase") not in ("done", "error") and not K.process_alive("lab-export"):
                st = dict(st, phase="error", error=st.get("error") or "the export process stopped; see the log")
            out[fmt] = st
    return out


def start_export(name, fmt):
    if fmt not in EXPORT_FORMATS:
        raise ValueError("format must be one of " + ", ".join(EXPORT_FORMATS))
    rd = run_dir(name)
    adapter = os.path.join(rd, "adapter")
    if not os.path.exists(os.path.join(adapter, "adapter_config.json")):
        raise ValueError("that run has no saved adapter")
    if K.process_alive("lab-export"):
        raise ValueError("another export is running; one at a time")
    gpu = side_gpu()
    stop_infer()                                   # exports need the GPU memory
    out = export_dir(name, fmt)
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out)
    job = K.read_json(os.path.join(rd, "job.json"))
    K.write_json(os.path.join(out, "job.json"), {"adapter": adapter, "format": fmt, "export_dir": out,
                                                 "max_seq_length": job["max_seq_length"]})
    K.write_json(os.path.join(out, "status.json"), {"phase": "starting", "started": time.time()})
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED="1")
    K.spawn("lab-export", [sys.executable, os.path.join(HERE, "export_unsloth.py"), os.path.join(out, "job.json")],
            os.path.join(LOGS, "unsloth_export.log"), env=env, cwd=HERE)
    return {"run": name, "format": fmt, "state": "starting"}


def artifact_dir(name, what):
    if what == "adapter":
        d = os.path.join(run_dir(name), "adapter")
    elif what in EXPORT_FORMATS:
        d = os.path.join(export_dir(name, what), "model")
        if (K.read_json(os.path.join(export_dir(name, what), "status.json"), {}) or {}).get("phase") != "done":
            raise ValueError("export %s is not finished; start it with export_run" % what)
    else:
        raise ValueError("what must be adapter or one of " + ", ".join(EXPORT_FORMATS))
    if not os.path.isdir(d):
        raise ValueError("nothing saved yet for %s" % what)
    return d


def push(name, repo_id, what="adapter", private=True):
    if not HF_TOKEN:
        raise ValueError("add an HF_TOKEN Kaggle secret with write access and re-run the server cell")
    if not HUB_ID.match(str(repo_id or "")):
        raise ValueError("repo_id must look like username/model-name")
    folder = artifact_dir(name, what)
    PUSH[name] = {"state": "uploading", "what": what, "repo_id": repo_id}

    def go():
        try:
            from huggingface_hub import HfApi
            api = HfApi(token=HF_TOKEN)
            api.create_repo(repo_id, private=bool(private), exist_ok=True)
            api.upload_folder(folder_path=folder, repo_id=repo_id, commit_message="Unsloth Lab run %s (%s)" % (name, what))
            PUSH[name] = {"state": "done", "what": what, "url": "https://huggingface.co/%s" % repo_id}
        except Exception as e:
            log.error("push failed\n%s", traceback.format_exc())
            PUSH[name] = {"state": "error", "what": what, "error": str(e)[:500]}
    threading.Thread(target=go, daemon=True).start()
    return PUSH[name]


def info():
    return {"gpus": K.gpu_info(), "base_models": BASE_MODELS, "export_formats": EXPORT_FORMATS, "hf_token": bool(HF_TOKEN),
            "dataset": dataset_info(), "train": train_status(with_log=False), "chat": chat_status(),
            "export_running": K.process_alive("lab-export")}


# ====================================================================== app
def build_app():
    app = K.App("Unsloth Fine-tuning Lab", password=os.environ.get("APP_PASSWORD"), log=log, max_body=1 << 30,
                instructions=(
                    "Fine-tunes small LLMs with Unsloth (4-bit LoRA). Typical flow: add_examples (or import_hf_dataset) "
                    "-> dataset_info -> start_training -> training_status until state is 'done' -> chat with "
                    "model='run:<name>' -> export_run (gguf_q4_k_m for Ollama/llama.cpp) -> export_status -> push_to_hub. "
                    "Long operations return immediately; poll their status tools. chat returns loading=true while "
                    "a model loads: call it again after ~30-60 s."))
    app.page("/", os.path.join(HERE, "lab_ui.html"))
    app.static("/static/", HERE)

    @app.route("GET", "/api/info")
    def _info(req):
        return info()

    # ---------------- dataset
    @app.route("GET", "/api/dataset")
    def _dataset(req):
        off, lim, q = req.arg("offset", 0, int), min(200, req.arg("limit", 50, int)), (req.arg("q", "") or "").lower()
        with LOCK:
            rows = list(enumerate(DATA["records"]))
        if q:
            rows = [(i, r) for i, r in rows if q in json.dumps(r, ensure_ascii=False).lower()]
        return {"total": len(rows), "rows": [{"index": i, "preview": D.preview(r), "kind": "text" if "text" in r else
                                              "chat", "chars": D.chars(r)} for i, r in rows[off:off + lim]]}

    @app.route("GET", "/api/dataset/record")
    def _record(req):
        i = req.arg("index", cast=int)
        with LOCK:
            if i is None or not 0 <= i < len(DATA["records"]):
                raise ValueError("no record %s" % i)
            return DATA["records"][i]

    @app.route("POST", "/api/dataset/upload")
    def _upload(req):
        recs, errs = D.parse(req.body().decode("utf-8-sig", "replace"), req.arg("format", "auto"))
        total = add_records(recs, req.arg("mode", "append")) if recs else len(DATA["records"])
        return {"added": len(recs), "total": total, "errors": errs[:30], "error_count": len(errs)}

    @app.route("POST", "/api/dataset/add")
    def _add(req):
        return t_add(**req.json())

    @app.route("POST", "/api/dataset/import_hf")
    def _import(req):
        b = req.json()
        return import_hf(b.get("name"), b.get("split") or "train", b.get("config"), b.get("max_rows", 5000),
                         b.get("mode", "append"))

    @app.route("POST", "/api/dataset/delete")
    def _delete_rows(req):
        idx = sorted({int(i) for i in req.json().get("indices", [])}, reverse=True)
        with LOCK:
            for i in idx:
                if 0 <= i < len(DATA["records"]):
                    del DATA["records"][i]
            save_dataset()
            return {"total": len(DATA["records"])}

    @app.route("POST", "/api/dataset/clear")
    def _clear(req):
        return {"total": add_records([], "replace")}

    @app.route("GET", "/api/dataset/download")
    def _download_ds(req):
        with LOCK:
            save_dataset()
        return K.FileResponse(p("dataset.jsonl"), name="dataset.jsonl", ctype="application/x-ndjson")

    # ---------------- training and runs
    @app.route("POST", "/api/train")
    def _train(req):
        return start_training(req.json())

    @app.route("GET", "/api/train/status")
    def _train_status(req):
        return train_status()

    @app.route("POST", "/api/train/stop")
    def _stop(req):
        return stop_training(bool(req.json().get("force")))

    @app.route("GET", "/api/runs")
    def _runs(req):
        return {"runs": list_runs()}

    @app.route("GET", r"/api/runs/(?P<name>[\w.-]+)")
    def _run(req):
        return run_detail(req.params["name"])

    @app.route("POST", r"/api/runs/(?P<name>[\w.-]+)/delete")
    def _delrun(req):
        return delete_run(req.params["name"])

    @app.route("POST", r"/api/runs/(?P<name>[\w.-]+)/export")
    def _export(req):
        return start_export(req.params["name"], req.json().get("format"))

    @app.route("GET", r"/api/runs/(?P<name>[\w.-]+)/download/(?P<what>[\w]+)")
    def _download(req):
        name, what = req.params["name"], req.params["what"]
        d = artifact_dir(name, what)
        files = [(os.path.join(dp, f), os.path.join("%s-%s" % (name, what), os.path.relpath(os.path.join(dp, f), d)))
                 for dp, dn, fn in os.walk(d) for f in sorted(fn)]
        if what == "adapter":
            rd = run_dir(name)
            files += [(os.path.join(rd, f), "%s-adapter/%s" % (name, f)) for f in ("job.json", "metrics.json",
                                                                                  "samples.json")]
        return K.ZipResponse(files, "%s-%s.zip" % (name, what))

    @app.route("POST", r"/api/runs/(?P<name>[\w.-]+)/push")
    def _push(req):
        b = req.json()
        return push(req.params["name"], b.get("repo_id"), b.get("what") or "adapter", b.get("private", True))

    # ---------------- chat
    @app.route("POST", "/api/chat")
    def _chat(req):
        b = req.json()
        return chat(b.get("model"), b.get("messages") or [], b.get("max_new_tokens", 512), b.get("temperature", 0.7))

    @app.route("POST", "/api/chat/unload")
    def _unload(req):
        stop_infer()
        return {"state": "off"}

    # ---------------- MCP tools
    @app.tool("list_base_models", "Suggested base models (any Hugging Face id also works) and the GPUs.")
    def t_bases():
        return {"base_models": BASE_MODELS, "gpus": K.gpu_info()}

    @app.tool("dataset_info", "Size and shape of the training dataset, plus the status of a Hub import.")
    def t_info():
        return dataset_info()

    @app.tool("add_examples", "Add training examples. Each record: {messages:[{role,content}...]}, "
              "{instruction, input?, output}, {prompt, completion} or {text}. mode=replace starts over.", {
                  "records": {"type": "array", "items": {"type": "object"}},
                  "mode": {"type": "string", "enum": ["append", "replace"], "default": "append"}}, ["records"])
    def t_add(records, mode="append"):
        if not isinstance(records, list):
            raise ValueError("records must be a list")
        recs, errs = [], []
        for i, r in enumerate(records):
            try:
                recs.append(D.normalize(r))
            except ValueError as e:
                errs.append({"index": i, "error": str(e)})
        total = add_records(recs, mode) if recs or mode == "replace" else len(DATA["records"])
        return {"added": len(recs), "total": total, "errors": errs[:30]}

    @app.tool("import_hf_dataset", "Import rows from a Hugging Face dataset (runs in the background; see dataset_info).", {
        "name": {"type": "string", "description": "owner/dataset, e.g. mlabonne/FineTome-100k"},
        "split": {"type": "string", "default": "train"}, "config": {"type": "string"},
        "max_rows": {"type": "integer", "default": 5000},
        "mode": {"type": "string", "enum": ["append", "replace"], "default": "append"}}, ["name"])
    def t_import(name, split="train", config=None, max_rows=5000, mode="append"):
        return import_hf(name, split, config, max_rows, mode)

    @app.tool("start_training", "Start a 4-bit LoRA fine-tune on the current dataset. Returns at once; poll "
              "training_status.", {
                  "name": {"type": "string"},
                  "base_model": {"type": "string", "description": "Hub id, or run:<name> to continue a run"},
                  "epochs": {"type": "number", "default": 2}, "max_steps": {"type": "integer", "default": 0,
                                                                           "description": ">0 overrides epochs"},
                  "learning_rate": {"type": "number", "default": 0.0002}, "lora_r": {"type": "integer", "default": 16},
                  "lora_alpha": {"type": "integer", "default": 16},
                  "max_seq_length": {"type": "integer", "default": 2048},
                  "batch_size": {"type": "integer", "default": 2}, "grad_accum": {"type": "integer", "default": 4},
                  "val_frac": {"type": "number", "default": 0.05}})
    def t_train(**o):
        return start_training(o)

    @app.tool("training_status", "Progress of the current or last run: step, loss, eval loss, ETA and the log tail. "
              "wait_seconds (max 85) waits for it to finish.", {"wait_seconds": {"type": "number", "default": 0}})
    def t_status(wait_seconds=0):
        t = time.time()
        while True:
            st = train_status()
            if st["state"] != "running" or time.time() - t >= min(MAX_WAIT, float(wait_seconds or 0)):
                st.pop("losses", None)
                return st
            time.sleep(3)

    @app.tool("stop_training", "Stop training after the current step and save the adapter trained so far.",
              {"force": {"type": "boolean", "default": False, "description": "kill at once without saving"}})
    def t_stop(force=False):
        return stop_training(force)

    @app.tool("list_runs", "All runs with metrics, exports and Hub pushes.")
    def t_runs():
        return list_runs()

    @app.tool("get_run", "One run: settings, status, metrics and sample answers on held-out examples.",
              {"name": {"type": "string"}}, ["name"])
    def t_run(name):
        return run_detail(name)

    @app.tool("chat", "Chat with a base model (Hub id) or a fine-tuned run (run:<name>). Returns loading=true while "
              "the model loads; call again after 30-60 s.", {
                  "model": {"type": "string"},
                  "messages": {"type": "array", "items": {"type": "object"},
                               "description": "[{role, content}], or pass a plain string as the user message"},
                  "max_new_tokens": {"type": "integer", "default": 512}, "temperature": {"type": "number", "default": 0.7}},
              ["model", "messages"])
    def t_chat(model, messages, max_new_tokens=512, temperature=0.7):
        return chat(model, messages, max_new_tokens, temperature)

    @app.tool("export_run", "Export a run: merged_16bit (full HF model), gguf_q4_k_m / gguf_q8_0 / gguf_f16 "
              "(llama.cpp, Ollama, LM Studio). Runs in the background; poll export_status.", {
                  "name": {"type": "string"}, "format": {"type": "string", "enum": EXPORT_FORMATS}}, ["name", "format"])
    def t_export(name, format):
        return start_export(name, format)

    @app.tool("export_status", "Status of a run's exports, with download paths.", {"name": {"type": "string"}}, ["name"])
    def t_export_status(name):
        run_dir(name)
        ex = export_list(name)
        for fmt, st in ex.items():
            if st.get("phase") == "done":
                st["download"] = "/api/runs/%s/download/%s" % (name, fmt)
        return {"exports": ex, "adapter_download": "/api/runs/%s/download/adapter" % name}

    @app.tool("push_to_hub", "Upload a run's adapter or a finished export to the Hugging Face Hub (needs HF_TOKEN "
              "with write access).", {
                  "name": {"type": "string"}, "repo_id": {"type": "string"},
                  "what": {"type": "string", "enum": ["adapter"] + EXPORT_FORMATS, "default": "adapter"},
                  "private": {"type": "boolean", "default": True}}, ["name", "repo_id"])
    def t_push(name, repo_id, what="adapter", private=True):
        return push(name, repo_id, what, private)

    @app.tool("delete_run", "Delete a run and its exports.", {"name": {"type": "string"}}, ["name"])
    def t_delete(name):
        return delete_run(name)

    return app


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7860)
    os.makedirs(p("runs"), exist_ok=True)
    os.makedirs(EXPORTS, exist_ok=True)
    load_dataset()
    if not os.path.exists(p("dataset.jsonl")):
        save_dataset()
    build_app().serve_forever(ap.parse_args().port)
''')
print("wrote", os.path.join(APP_DIR, 'lab_server.py'))

In [ ]:
# Lab page
# Writes lab_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/unsloth_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'lab_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Unsloth Lab</title>
<link rel="stylesheet" href="/static/kit.css">
<style>
.form{display:grid;gap:10px;grid-template-columns:repeat(auto-fill,minmax(150px,1fr))}
#chart{width:100%;height:180px;background:var(--panel2);border-radius:8px}
.msgs{display:flex;flex-direction:column;gap:8px;max-height:55vh;overflow:auto;padding:4px}
.msg{padding:9px 12px;border-radius:10px;max-width:85%;white-space:pre-wrap}
.msg.user{align-self:flex-end;background:var(--accent);color:var(--accent-ink)}
.msg.assistant{align-self:flex-start;background:var(--panel2);border:1px solid var(--line)}
.sample{border:1px solid var(--line);border-radius:8px;padding:8px;margin-bottom:8px;font-size:13px}
td .row{gap:6px}
</style></head><body>
<header class="top"><h1>Unsloth Fine-tuning Lab</h1><span class="sub" id="gpus"></span>
  <div class="right small muted" id="status"></div></header>
<nav class="tabs"><button data-tab="data">1 · Dataset</button><button data-tab="train">2 · Train</button><button data-tab="runs">3 · Runs &amp; export</button><button data-tab="chat">4 · Chat</button><button data-tab="agents">Agents</button></nav>
<main>
<section data-pane="data">
  <div class="grid">
    <div class="card"><h2>Upload</h2>
      <div class="drop" id="drop">Drop a .jsonl / .json / .csv file, or click</div><input type="file" id="file" accept=".jsonl,.json,.csv,.txt" hidden>
      <label class="inline" style="margin-top:8px"><input type="checkbox" id="replace"> Replace the current dataset</label>
      <p class="small muted">One record per line, any of: <code>{"messages":[{"role":"user","content":"…"},{"role":"assistant","content":"…"}]}</code>, ShareGPT <code>{"conversations":[{"from":"human","value":"…"}…]}</code>, Alpaca <code>{"instruction","input","output"}</code>, <code>{"prompt","completion"}</code>, or <code>{"text"}</code>.</p>
    </div>
    <div class="card"><h2>From the Hugging Face Hub</h2>
      <div class="col"><input id="hfname" placeholder="owner/dataset, e.g. mlabonne/FineTome-100k">
      <div class="row"><input id="hfsplit" value="train" style="width:90px"><input id="hfconfig" placeholder="config (optional)" style="width:150px"><input id="hfmax" type="number" value="2000" style="width:100px"></div>
      <button id="hfgo">Import</button><div class="small muted" id="hfstate"></div></div>
    </div>
  </div>
  <div class="card"><div class="row"><h2 style="margin:0">Examples</h2><span class="muted small" id="dsstats"></span>
    <span style="margin-left:auto" class="row"><input id="q" placeholder="Search"><a class="btn" href="/api/dataset/download">Download</a><button class="danger" id="clear">Clear</button></span></div>
    <table><thead><tr><th>#</th><th>Example</th><th>Type</th><th></th></tr></thead><tbody id="rows"></tbody></table>
    <div class="row" style="margin-top:8px"><button id="prev">←</button><span id="page" class="small muted"></span><button id="next">→</button></div>
  </div>
</section>
<section data-pane="train">
  <div class="card"><h2>New run</h2>
    <div class="form">
      <label style="grid-column:span 2">Base model<select id="base"></select></label>
      <label style="grid-column:span 2">…or any Hub id / run:&lt;name&gt;<input id="basecustom" placeholder="optional"></label>
      <label>Run name<input id="rname" placeholder="auto"></label>
      <label>Epochs<input id="epochs" type="number" step="0.5" value="2"></label>
      <label>Max steps (0 = epochs)<input id="max_steps" type="number" value="0"></label>
      <label>Learning rate<input id="learning_rate" value="0.0002"></label>
      <label>LoRA rank<input id="lora_r" type="number" value="16"></label>
      <label>LoRA alpha<input id="lora_alpha" type="number" value="16"></label>
      <label>Max sequence length<input id="max_seq_length" type="number" value="2048"></label>
      <label>Batch size<input id="batch_size" type="number" value="2"></label>
      <label>Grad accumulation<input id="grad_accum" type="number" value="4"></label>
      <label>Validation share<input id="val_frac" value="0.05"></label>
    </div>
    <div class="row" style="margin-top:12px"><button class="primary" id="start">Start training</button><button id="stop">Stop (save adapter)</button><span class="small muted">7–8B models on a T4: keep max sequence length ≤ 2048 and batch size 1–2.</span></div>
  </div>
  <div class="card"><div class="row"><h2 style="margin:0">Current run</h2><span id="trun"></span></div>
    <div id="tinfo" class="small muted" style="margin:8px 0"></div><div id="tbar"></div>
    <canvas id="chart"></canvas>
    <h3>Log</h3><pre class="log" id="tlog"></pre></div>
</section>
<section data-pane="runs"><div class="card"><h2>Runs</h2><div id="runs"></div></div><div class="card" id="rundetail" hidden></div></section>
<section data-pane="chat">
  <div class="card"><div class="row"><label>Model<select id="cmodel"></select></label><label>Max new tokens<input id="cmax" type="number" value="512" style="width:90px"></label><label>Temperature<input id="ctemp" value="0.7" style="width:70px"></label>
    <button id="creset" style="align-self:flex-end">New chat</button><span class="small muted" id="cstate" style="align-self:flex-end"></span></div></div>
  <div class="card"><div class="msgs" id="msgs"></div>
    <form id="cf" class="row" style="margin-top:10px"><textarea id="cin" rows="2" style="flex:1;min-height:44px" placeholder="Message (Enter to send)"></textarea><button class="primary" id="csend">Send</button></form></div>
</section>
<section data-pane="agents"><div class="card"><h2>MCP server</h2><pre class="log" id="mcp"></pre>
  <p class="small muted">Tools: list_base_models, dataset_info, add_examples, import_hf_dataset, start_training, training_status, stop_training, list_runs, get_run, chat, export_run, export_status, push_to_hub, delete_run. The same tools answer REST calls at <code>POST /mcp/tools/&lt;name&gt;</code>.</p></div></section>
</main>
<script src="/static/kit.js"></script>
<script>
let INFO = null, off = 0, chatMsgs = [];
$("#mcp").textContent = mcpHelp("unsloth-lab");
setupTabs(t => { if (t === "runs") loadRuns(); if (t === "data") loadRows(); });

async function loadInfo() {
  INFO = await api("/api/info");
  $("#gpus").textContent = INFO.gpus.map(g => "GPU" + g.index + " " + (g.mem_used_mb / 1024).toFixed(1) + "/" + (g.mem_total_mb / 1024).toFixed(0) + " GB").join(" · ") || "no GPU";
  const ds = INFO.dataset;
  $("#dsstats").textContent = ds.count ? ds.count + " examples · median " + ds.chars_median + " chars · ~" + ds.approx_tokens_total.toLocaleString() + " tokens" : "empty";
  const imp = ds.import_job || {};
  $("#hfstate").textContent = imp.state === "running" ? "Importing " + imp.name + "…" : imp.state === "done" ? "Imported " + imp.added + " rows (" + imp.skipped + " skipped)" : imp.state === "error" ? "Import failed: " + imp.error : "";
  $("#status").textContent = "training: " + (INFO.train.state || "idle") + " · chat: " + INFO.chat.state + (INFO.export_running ? " · exporting" : "");
  if (!$("#base").options.length) INFO.base_models.forEach(b => $("#base").append(el("option", {value: b.id}, b.label)));
}

// ---------------- dataset
async function loadRows() {
  const d = await api("/api/dataset?limit=50&offset=" + off + "&q=" + encodeURIComponent($("#q").value));
  $("#rows").replaceChildren(...d.rows.map(r => el("tr", {}, el("td", {class: "muted"}, r.index), el("td", {}, r.preview), el("td", {}, el("span", {class: "badge"}, r.kind)),
    el("td", {}, el("button", {class: "danger", onclick: async () => { await api("/api/dataset/delete", {json: {indices: [r.index]}}); loadRows(); loadInfo(); }}, "×")))));
  $("#page").textContent = d.total ? (off + 1) + "–" + Math.min(off + 50, d.total) + " of " + d.total : "No examples yet.";
}
$("#prev").onclick = () => { off = Math.max(0, off - 50); loadRows(); };
$("#next").onclick = () => { off += 50; loadRows(); };
$("#q").oninput = () => { off = 0; loadRows(); };
$("#clear").onclick = async () => { if (confirm("Remove every example?")) { await api("/api/dataset/clear", {json: {}}); loadRows(); loadInfo(); } };
dropZone($("#drop"), $("#file"), async files => {
  const f = files[0], fmt = f.name.endsWith(".csv") ? "csv" : f.name.endsWith(".json") ? "json" : "auto";
  const r = await guard(() => api("/api/dataset/upload?format=" + fmt + "&mode=" + ($("#replace").checked ? "replace" : "append"), {body: f}));
  toast("Added " + r.added + " examples" + (r.error_count ? ", " + r.error_count + " skipped (first: line " + r.errors[0].line + ": " + r.errors[0].error + ")" : ""), r.error_count ? "bad" : "ok");
  loadRows(); loadInfo();
});
$("#hfgo").onclick = () => guard(() => api("/api/dataset/import_hf", {json: {name: $("#hfname").value.trim(), split: $("#hfsplit").value, config: $("#hfconfig").value || null, max_rows: +$("#hfmax").value}}), "Import started").then(loadInfo);

// ---------------- training
$("#start").onclick = async () => {
  const o = {base_model: $("#basecustom").value.trim() || $("#base").value, name: $("#rname").value.trim()};
  ["epochs", "max_steps", "learning_rate", "lora_r", "lora_alpha", "max_seq_length", "batch_size", "grad_accum", "val_frac"].forEach(k => o[k] = $("#" + k).value);
  await guard(() => api("/api/train", {json: o}), "Training started"); loadTrain();
};
$("#stop").onclick = () => guard(() => api("/api/train/stop", {json: {}})).then(r => toast(r.note || r.reason));

function drawChart(st) {
  const c = $("#chart"), ctx = c.getContext("2d"), W = c.width = c.clientWidth * devicePixelRatio, H = c.height = c.clientHeight * devicePixelRatio;
  ctx.clearRect(0, 0, W, H);
  const L = st.losses || [], E = st.evals || [];
  if (L.length < 2) return;
  const all = L.concat(E), maxS = Math.max(...all.map(x => x[0])), ys = all.map(x => x[1]), lo = Math.min(...ys), hi = Math.max(...ys);
  const X = s => 30 + (W - 40) * s / (maxS || 1), Y = v => 10 + (H - 30) * (1 - (v - lo) / ((hi - lo) || 1));
  const css = getComputedStyle(document.documentElement);
  [[L, css.getPropertyValue("--accent")], [E, css.getPropertyValue("--warn")]].forEach(([pts, col]) => {
    if (!pts.length) return; ctx.strokeStyle = col.trim(); ctx.lineWidth = 2 * devicePixelRatio; ctx.beginPath();
    pts.forEach(([s, v], i) => i ? ctx.lineTo(X(s), Y(v)) : ctx.moveTo(X(s), Y(v))); ctx.stroke(); });
  ctx.fillStyle = css.getPropertyValue("--muted").trim(); ctx.font = (11 * devicePixelRatio) + "px system-ui";
  ctx.fillText(hi.toFixed(3), 2, 14 * devicePixelRatio); ctx.fillText(lo.toFixed(3), 2, H - 18);
}
async function loadTrain() {
  const st = await api("/api/train/status");
  $("#trun").replaceChildren(st.run ? el("span", {class: "row"}, el("b", {}, st.run), badge(st.state)) : el("span", {class: "muted"}, "No runs yet"));
  const bits = [];
  if (st.phase) bits.push(st.phase);
  if (st.total_steps) bits.push("step " + (st.step || 0) + "/" + st.total_steps);
  if (st.loss != null) bits.push("loss " + st.loss);
  if (st.eval_loss != null) bits.push("eval " + st.eval_loss);
  if (st.eta != null && st.state === "running") bits.push("about " + fmtDur(st.eta) + " left");
  if (st.error) bits.push("error: " + st.error);
  $("#tinfo").textContent = bits.join(" · ");
  $("#tbar").replaceChildren(st.total_steps ? progressBar((st.step || 0) / st.total_steps) : "");
  $("#tlog").textContent = st.log || "";
  drawChart(st);
}

// ---------------- runs
async function loadRuns() {
  const {runs} = await api("/api/runs");
  const box = $("#runs");
  if (!runs.length) { box.replaceChildren(el("div", {class: "empty"}, "No runs yet.")); return; }
  box.replaceChildren(el("table", {}, el("thead", {}, el("tr", {}, ...["Run", "Base", "State", "Loss", "Exports", ""].map(h => el("th", {}, h)))),
    el("tbody", {}, ...runs.map(r => el("tr", {},
      el("td", {}, el("a", {href: "#runs", onclick: () => showRun(r.name)}, r.name), el("div", {class: "small muted"}, fmtAgo(r.created) + " · " + r.examples + " examples")),
      el("td", {class: "small"}, r.base), el("td", {}, badge(r.state)),
      el("td", {class: "small"}, r.metrics ? (r.metrics.final_loss ?? "") + (r.metrics.eval_loss != null ? " / eval " + r.metrics.eval_loss : "") : ""),
      el("td", {class: "small"}, ...Object.entries(r.exports).map(([f, s]) => el("div", {}, f + ": ", s.phase === "done" ? el("a", {href: "/api/runs/" + r.name + "/download/" + f}, "download") : s.phase))),
      el("td", {}, el("div", {class: "row"},
        r.has_adapter ? el("a", {class: "btn", href: "/api/runs/" + r.name + "/download/adapter"}, "Adapter .zip") : null,
        r.has_adapter ? el("select", {onchange: async e => { const f = e.target.value; e.target.value = ""; if (f) { await guard(() => api("/api/runs/" + r.name + "/export", {json: {format: f}}), "Export started"); loadRuns(); } }},
          el("option", {value: ""}, "Export…"), ...INFO.export_formats.map(f => el("option", {value: f}, f))) : null,
        r.has_adapter && INFO.hf_token ? el("button", {onclick: async () => { const repo = prompt("Hub repo id (username/name):"); if (!repo) return;
          const what = prompt("What to push: adapter, or a finished export (" + INFO.export_formats.join(", ") + ")", "adapter");
          await guard(() => api("/api/runs/" + r.name + "/push", {json: {repo_id: repo, what}}), "Upload started"); }}, "Push") : null,
        r.push ? el("span", {class: "small muted"}, "Hub: " + (r.push.url || r.push.state)) : null,
        el("button", {class: "danger", onclick: async () => { if (confirm("Delete run " + r.name + "?")) { await guard(() => api("/api/runs/" + r.name + "/delete", {json: {}})); loadRuns(); } }}, "×"))))))));
}
async function showRun(name) {
  const r = await api("/api/runs/" + name), d = $("#rundetail");
  d.hidden = false;
  d.replaceChildren(el("h2", {}, "Run " + name), el("pre", {class: "log"}, JSON.stringify({settings: r.job, metrics: r.metrics}, null, 1)),
    el("h3", {}, "Answers on held-out examples"), ...(r.samples.length ? r.samples.map(s => el("div", {class: "sample"},
      el("div", {}, el("b", {}, "Prompt: "), s.prompt), el("div", {}, el("b", {}, "Expected: "), s.expected), el("div", {}, el("b", {}, "Model: "), s.model))) : [el("div", {class: "muted small"}, "None (needs chat data and a validation share).")]));
}

// ---------------- chat
async function fillModels() {
  const {runs} = await api("/api/runs"), sel = $("#cmodel"), cur = sel.value;
  sel.replaceChildren(...runs.filter(r => r.has_adapter).map(r => el("option", {value: "run:" + r.name}, "run: " + r.name)),
    ...INFO.base_models.map(b => el("option", {value: b.id}, b.label)));
  if (cur) sel.value = cur;
}
function renderMsgs() { $("#msgs").replaceChildren(...chatMsgs.map(m => el("div", {class: "msg " + m.role}, m.content))); $("#msgs").scrollTop = 1e9; }
$("#creset").onclick = () => { chatMsgs = []; renderMsgs(); };
$("#cin").addEventListener("keydown", e => { if (e.key === "Enter" && !e.shiftKey) { e.preventDefault(); $("#cf").requestSubmit(); } });
$("#cf").addEventListener("submit", async e => {
  e.preventDefault();
  const text = $("#cin").value.trim(); if (!text) return;
  $("#cin").value = ""; chatMsgs.push({role: "user", content: text}); renderMsgs();
  $("#csend").disabled = true;
  try {
    for (let i = 0; i < 40; i++) {
      const r = await api("/api/chat", {json: {model: $("#cmodel").value, messages: chatMsgs, max_new_tokens: +$("#cmax").value, temperature: +$("#ctemp").value}});
      if (!r.loading) { chatMsgs.push({role: "assistant", content: r.text}); $("#cstate").textContent = r.tokens + " tokens in " + r.seconds + " s"; break; }
      $("#cstate").textContent = r.message; await new Promise(res => setTimeout(res, 8000));
    }
  } catch (err) { toast(err.message, "bad"); chatMsgs.pop(); }
  renderMsgs(); $("#csend").disabled = false;
});

loadInfo().then(() => { loadRows(); fillModels(); });
every(5000, loadInfo);
every(4000, () => $("[data-pane=train]").classList.contains("on") ? loadTrain() : null);
every(15000, () => $("[data-pane=chat]").classList.contains("on") ? fillModels() : null);
every(10000, () => $("[data-pane=runs]").classList.contains("on") ? loadRuns() : null);
</script></body></html>
''')
print("wrote", os.path.join(APP_DIR, 'lab_ui.html'))

In [ ]:
# 3. Load the helpers
import os, sys
APP_DIR = '/kaggle/working/unsloth_app'
_missing = [f for f in ("studio_http.py", "kit.css", "kit.js") if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("Run the 'Writes ... into APP_DIR' cells above first (missing: %s)" % ", ".join(_missing))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import studio_http as K   # Notebook helpers: kaggle_secret, spawn, wait_http, tail, start_tunnel, keep_alive

In [ ]:
# 4. Install Unsloth (quiet; 3-5 minutes). Unsloth picks the PyTorch / xformers / bitsandbytes versions it
#    needs, so this notebook never imports torch itself: training, chat and exports run in their own processes.
import subprocess
def pip_install(*args, keep_core=True):
    """pip install, quietly. keep_core pins Kaggle's own torch / numpy builds so nothing replaces them."""
    cons = "/tmp/keep_core.txt"
    lines = []
    if keep_core:
        import importlib.metadata as md
        for pkg in ("torch", "torchvision", "torchaudio", "numpy"):
            try:
                lines.append("%s==%s" % (pkg, md.version(pkg)))
            except md.PackageNotFoundError:
                pass
    open(cons, "w").write("\n".join(lines) + "\n")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons, *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(args))

os.makedirs(LOG_DIR, exist_ok=True)
print("Unsloth...")
pip_install("unsloth", keep_core=False)
pip_install("huggingface_hub", "datasets", keep_core=False)
K.ensure_cloudflared()
check = subprocess.run([sys.executable, "-c", "import unsloth, trl, transformers, torch; "
                        "print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| transformers', "
                        "transformers.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"],
                       capture_output=True, text=True, timeout=600)
print(check.stdout.strip().splitlines()[-1] if check.returncode == 0 else
      "Unsloth could not load (it needs a GPU accelerator):\n" + check.stderr[-1500:])
gpus = K.gpu_info()
print("GPUs:", ", ".join("%d: %s" % (g["index"], g["name"]) for g in gpus) or "none - pick 'GPU T4 x2' in the settings")

In [ ]:
# 5. Start the lab on port 7860 (background process). Log: /kaggle/working/logs/unsloth_lab.log
PASSWORD = K.ui_password("UNSLOTH_UI_PASSWORD")
HF_TOKEN = K.kaggle_secret("HF_TOKEN")
env = dict(os.environ, APP_PASSWORD=PASSWORD, WORK_DIR=WORK_DIR, EXPORT_DIR=EXPORT_DIR, LOG_DIR=LOG_DIR,
           APP_LOG=LOG_DIR + "/unsloth_lab.log", PYTHONUNBUFFERED="1")
env.pop("HF_TOKEN", None)
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN
else:
    print("No HF_TOKEN secret: gated models (Llama) and 'Push to Hub' are unavailable; everything else works.")
K.spawn("unsloth-lab", [sys.executable, os.path.join(APP_DIR, "lab_server.py"), "--port", str(PORT)],
        LOG_DIR + "/unsloth_lab.log", env=env, cwd=APP_DIR)
if not K.wait_http("http://127.0.0.1:%d/health" % PORT, timeout=60, name="unsloth-lab"):
    print(K.tail(LOG_DIR + "/unsloth_lab.log", 40))
    raise RuntimeError("The lab did not start; log above")
for host in ("127.0.0.1", "[::1]"):
    print(host, "->", "up" if K.wait_http("http://%s:%d/health" % (host, PORT), timeout=3) else "not reachable")
print("Training, chat and export processes keep running if you re-run this cell.")

In [ ]:
# 6. Publish through your Cloudflare Tunnel (token from the UNSLOTH_TUNNEL_TOKEN secret, never printed)
tunnel = K.start_tunnel("UNSLOTH_TUNNEL_TOKEN", PORT, LOG_DIR + "/cloudflared.log")

In [ ]:
# 7. Keep-alive monitor. Leave it running; interrupting it only stops the status lines.
def train_line():
    t = K.api_get(PORT, "/api/info", PASSWORD)["train"]
    if t.get("state") == "running":
        return "training %s step %s/%s loss %s" % (t.get("run"), t.get("step", 0), t.get("total_steps", "?"), t.get("loss", "-"))
    return "training=%s" % t.get("state", "idle")

K.keep_alive(PORT, ["unsloth-lab", "cloudflared"], extra=train_line)